# 综合实训 · 基于 Pthreads 的线程池及其应用

**所属**：《并行计算》第四章 · Pthreads 多线程编程　|　**难度**：⭐⭐⭐⭐⭐ 综合　|　**预计时长**：60–90 分钟

> **实验说明**
> 1. 本实验是第四章的**综合实训**。第四章前九个实验分别训练了线程创建与回收、互斥锁、条件变量、屏障、生产者—消费者、伪共享等单项技术；本实验将这些技术组装为一个可复用的构件——**线程池**，并用它完成一项真实的计算任务。
> 2. 实验的重点是**线程池本身**：任务负载选用 Mandelbrot 集渲染，因为它计算简单、结果可逐字节校验，且各像素的计算量天然差异悬殊，正适合暴露负载不均衡问题。
> 3. 本实验**一次给出完整的线程池实现**，随后通过四组对照实验回答四个问题：为什么需要池、为什么需要队列、任务粒度取多大、哪些写法是错的。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 本实验依赖 **POSIX 线程库**，编译时需添加 `-pthread`；对处理器架构无特殊要求，建议在华为鲲鹏处理器上运行以获得足够的核心数。
> 6. ⚠️ **第 10 节的两个故障注入版本将按设计崩溃或报告 `FAIL`**，这不是环境故障，而是本实验的核心教学素材，请勿跳过。


## 🎯 学习目标

完成本任务的学习后，学生应能够：

- 说明**线程池**要解决的两个问题——线程创建与回收的固定开销、以及任务数远大于核心数时的调度问题
- 独立实现一个线程池，正确使用**互斥锁**保护共享队列、使用**条件变量**在队列为空时挂起工作线程
- 解释条件变量必须配合 `while` 循环使用的原因，区分 `pthread_cond_signal` 与 `pthread_cond_broadcast` 的适用场合
- 区分线程池关闭的两种语义——**排空后退出（drain）**与**立即停止（stop-now）**，并说明判定条件书写顺序的差异如何决定语义
- 使用**每线程私有计数器**（配合缓存行填充）在无锁条件下采集负载统计，并用**不均衡因子** $\max/\mathrm{avg}$ 定量刻画负载分布
- 通过实验建立**任务粒度**的权衡模型：粒度过细时队列与锁的开销占主导，粒度过粗时负载不均衡占主导
- 通过故障注入实验，观察 `if` 误用于条件等待所导致的空指针解引用，以及关闭语义写反所导致的任务丢失
- 独立完成扩展实验：为线程池增加**有界队列**（背压）与**完成栅栏** `tp_wait()`


## 🗺️ 学习路径

1. **准备阶段**：回顾第四章已完成的单项实验——互斥锁、条件变量、生产者—消费者、伪共享，本实验是它们的综合应用
2. **线程池的实现**：先约定接口（`threadpool.h`），再给出实现（`threadpool.c`）。实现代码中标注了三处关键写法及其理由
3. **案例任务**：Mandelbrot 集渲染。明确任务的划分方式（像素区间）与校验方式（逐字节比对）
4. **对照实验一 · 四种方案**：串行 / 每任务创建线程（无池）/ 线程池·静态均分 / 线程池·动态队列
   → 第 2 行与第 4 行的差异回答"**为什么需要池**"，第 3 行与第 4 行的差异回答"**为什么需要队列**"
5. **负载均衡的量化**：读取每个工作线程的迭代次数与忙碌时间，计算不均衡因子，解释静态均分为何失效
6. **对照实验二 · 任务粒度**：固定线程数，扫描每任务像素数，观察耗时曲线的下凹形状
7. **故障注入**：分别把 `while` 改成 `if`、把关闭判定的顺序写反，观察两类不同的失效表现
8. **🚀 扩展实验**：为线程池增加有界队列与 `tp_wait()` 完成栅栏，并测量池复用的收益


## 1. 背景与动机：为什么需要线程池

第四章前面的实验中，并行程序的写法始终是"**一个任务一个线程**"：需要并行执行 $P$ 段工作，就调用 $P$ 次 `pthread_create`，再调用 $P$ 次 `pthread_join`。当 $P$ 等于核心数、且每段工作足够长时，这种写法完全可行。

一旦任务数量增多，这种写法会同时遇到两个问题。

<!--
| 问题 | 表现 | 根源 |
|---|---|---|
| **线程创建与回收的固定开销** | 每次 `pthread_create` 都要在内核中建立任务结构、分配并映射线程栈（默认 8 MiB 虚拟地址空间）；`pthread_join` 还要等待并回收。当任务本身只需运行几十微秒时，这些固定开销与任务本身同量级 | 线程是**内核资源**，不是轻量对象 |
| **并发度失控** | 若为 $N$ 个任务同时创建 $N$ 个线程，可运行线程数远超核心数，操作系统被迫频繁切换上下文；若分批创建，则每批之间必须 `join`，形成一次隐式同步，最慢的那个线程决定整批的耗时 | 缺少一个能"持续接收任务"的执行者集合 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">问题</th>
      <th style="text-align: left;">表现</th>
      <th style="text-align: left;">根源</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>线程创建与回收的固定开销</strong></td>
      <td style="text-align: left;">每次 <code>pthread_create</code> 都要在内核中建立任务结构、分配并映射线程栈（默认 8 MiB 虚拟地址空间）；<code>pthread_join</code> 还要等待并回收。当任务本身只需运行几十微秒时，这些固定开销与任务本身同量级</td>
      <td style="text-align: left;">线程是<strong>内核资源</strong>，不是轻量对象</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>并发度失控</strong></td>
      <td style="text-align: left;">若为 N 个任务同时创建 N 个线程，可运行线程数远超核心数，操作系统被迫频繁切换上下文；若分批创建，则每批之间必须 <code>join</code>，形成一次隐式同步，最慢的那个线程决定整批的耗时</td>
      <td style="text-align: left;">缺少一个能"持续接收任务"的执行者集合</td>
    </tr>
  </tbody>
</table>


**线程池**同时解决这两个问题，其思想可以用一句话概括：

> **线程只创建一次，任务反复提交。** $N$ 个工作线程在程序启动时创建，此后不断从一个共享的任务队列中取出任务执行，直到线程池被销毁。

由此，线程的创建开销从"每任务一次"降为"每程序一次"；同时，由于工作线程数固定为 $N$，无论提交多少任务，可运行线程数都不会超过 $N$，并发度天然受控。此外还获得一个重要的附带收益：**先完成任务的线程会立即领取下一个任务**，因此任务在线程间的分配是按实际完成速度自动进行的，这正是解决负载不均衡问题所需要的机制。

本实验要验证的正是这三点。


## 2. 线程池的结构

一个最小可用的线程池由三部分状态构成，缺一不可：

<!--
| 状态 | 类型 | 作用 |
|---|---|---|
| 任务队列 | 单向链表 `head` / `tail` | 存放已提交但尚未执行的任务。任务 = 函数指针 + 一个 `void*` 参数 |
| 互斥锁 | `pthread_mutex_t lock` | 保护队列与关闭标志。任何对 `head`、`tail`、`shutdown` 的读写都必须在持锁状态下进行 |
| 条件变量 | `pthread_cond_t not_empty` | 队列为空时让工作线程**挂起**而不是空转；有任务提交时将其唤醒 |
| 关闭标志 | `int shutdown` | 通知工作线程"不会再有新任务了，把队列排空后退出" |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">状态</th>
      <th style="text-align: left;">类型</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">任务队列</td>
      <td style="text-align: left;">单向链表 <code>head</code> / <code>tail</code></td>
      <td style="text-align: left;">存放已提交但尚未执行的任务。任务 = 函数指针 + 一个 <code>void*</code> 参数</td>
    </tr>
    <tr>
      <td style="text-align: left;">互斥锁</td>
      <td style="text-align: left;"><code>pthread_mutex_t lock</code></td>
      <td style="text-align: left;">保护队列与关闭标志。任何对 <code>head</code>、<code>tail</code>、<code>shutdown</code> 的读写都必须在持锁状态下进行</td>
    </tr>
    <tr>
      <td style="text-align: left;">条件变量</td>
      <td style="text-align: left;"><code>pthread_cond_t not_empty</code></td>
      <td style="text-align: left;">队列为空时让工作线程<strong>挂起</strong>而不是空转；有任务提交时将其唤醒</td>
    </tr>
    <tr>
      <td style="text-align: left;">关闭标志</td>
      <td style="text-align: left;"><code>int shutdown</code></td>
      <td style="text-align: left;">通知工作线程"不会再有新任务了，把队列排空后退出"</td>
    </tr>
  </tbody>
</table>


数据流向如下：

```
                 ┌──────────── 主线程（生产者）────────────┐
                 │  tp_submit(pool, fn, arg)              │
                 │    lock → 入队 → cond_signal → unlock  │
                 └──────────────────┬─────────────────────┘
                                    │
                        ┌───────────▼───────────┐
                        │   任务队列（共享）      │
                        │   head → t1 → t2 → …  │
                        └───────────┬───────────┘
                                    │
      ┌───────────────┬─────────────┼─────────────┬───────────────┐
      ▼               ▼             ▼             ▼               ▼
  worker 0        worker 1      worker 2      worker 3   …    worker N-1
  lock → 队空则 cond_wait → 出队 → unlock → 执行任务 → 循环
```

请特别注意流程末尾部分：**工作线程必须先释放互斥锁，再执行任务**。若持锁执行任务，则同一时刻只有一个线程能运行任务体，线程池将退化为串行执行。

### 接口设计

本实验采用如下接口。它刻意保持最小：仅四个函数，包括一个供任务体查询自身身份的辅助函数。

<!--
| 函数 | 语义 |
|---|---|
| `tp_create(nthreads)` | 创建线程池，启动 `nthreads` 个工作线程 |
| `tp_submit(pool, fn, arg)` | 把一个任务追加到队列尾部。**不阻塞**（队列无上限，见扩展实验） |
| `tp_destroy(pool)` | 把队列**排空**，回收全部工作线程，释放资源。返回时保证已提交的任务都已执行完毕 |
| `tp_worker_id()` | 在任务体内部调用时返回当前工作线程的编号 $0..N-1$，在池外调用返回 $-1$ |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">语义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>tp_create(nthreads)</code></td>
      <td style="text-align: left;">创建线程池，启动 <code>nthreads</code> 个工作线程</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tp_submit(pool, fn, arg)</code></td>
      <td style="text-align: left;">把一个任务追加到队列尾部。<strong>不阻塞</strong>（队列无上限，见扩展实验）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tp_destroy(pool)</code></td>
      <td style="text-align: left;">把队列<strong>排空</strong>，回收全部工作线程，释放资源。返回时保证已提交的任务都已执行完毕</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tp_worker_id()</code></td>
      <td style="text-align: left;">在任务体内部调用时返回当前工作线程的编号 0..N-1，在池外调用返回 -1</td>
    </tr>
  </tbody>
</table>


> **⭐ 关于 `tp_destroy` 的双重身份**
>
> 基础版线程池只有 `tp_destroy` 一个函数能提供"全部任务已完成"的保证，因此本实验中每完成一轮渲染都要销毁一次线程池。这显然不是理想的用法——**扩展实验**将补上一个专门的完成栅栏 `tp_wait()`，使线程池可以跨轮复用。


## 3. 条件变量的正确使用

线程池实现中有三处写法，初学者极易写错，而其中两处错误不会产生编译警告。本节先讲清原理，第 10 节再用故障注入验证。

### 3.1 必须使用 `while`，不能使用 `if`

```c
while (pool->head == NULL && !pool->shutdown)
    pthread_cond_wait(&pool->not_empty, &pool->lock);
```

`pthread_cond_wait` 返回，**并不等于**它所等待的条件已经成立。有两个原因：

1. **任务被别的线程抢先取走**。`tp_submit` 每提交一个任务发出一次 `signal`，被唤醒的线程需要重新竞争互斥锁；在它拿到锁之前，另一个刚执行完任务、正在循环回来的线程可能已经把队列取空；
2. **虚假唤醒（spurious wakeup）**。POSIX 标准明确允许 `pthread_cond_wait` 在无人发出信号的情况下返回。

因此，被唤醒后**必须重新检查判定条件**，这正是 `while` 的作用。若写成 `if`，线程将在队列为空时继续向下执行 `t = pool->head`，随即对空指针解引用。

### 3.2 关闭判定的书写顺序决定关闭语义

<!--
| 写法 | 语义 | 后果 |
|---|---|---|
| `if (head == NULL && shutdown) break;` | **排空后退出（drain）**：只有队列为空且已请求关闭时才退出 | 已提交的任务保证全部执行 |
| `if (shutdown) break;` | **立即停止（stop-now）**：一旦请求关闭立即退出 | 队列中尚未执行的任务被**静默丢弃** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">写法</th>
      <th style="text-align: left;">语义</th>
      <th style="text-align: left;">后果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>if (head == NULL && shutdown) break;</code></td>
      <td style="text-align: left;"><strong>排空后退出（drain）</strong>：只有队列为空且已请求关闭时才退出</td>
      <td style="text-align: left;">已提交的任务保证全部执行</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>if (shutdown) break;</code></td>
      <td style="text-align: left;"><strong>立即停止（stop-now）</strong>：一旦请求关闭立即退出</td>
      <td style="text-align: left;">队列中尚未执行的任务被<strong>静默丢弃</strong></td>
    </tr>
  </tbody>
</table>


两种语义在工程上都有其用途（例如服务优雅停机与强制终止），但**必须是有意选择的**。本实验采用 drain 语义，因为它与 `tp_destroy` 承担的"完成栅栏"职责一致。第 10 节将把这一行改写为 stop-now，观察渲染结果如何变得不完整。

### 3.3 `signal` 与 `broadcast` 的选择

<!--
| 位置 | 调用 | 理由 |
|---|---|---|
| `tp_submit` | `pthread_cond_signal` | 新增**一个**任务只能占用**一个**工作线程。唤醒全部空闲线程会造成"惊群"：所有线程醒来竞争同一把锁，只有一个能拿到任务，其余重新进入等待 |
| `tp_destroy` | `pthread_cond_broadcast` | 关闭标志必须被**每一个**空闲线程观察到。若只 `signal` 一次，其余线程将永远阻塞在条件变量上，`pthread_join` 随之永久挂起 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">位置</th>
      <th style="text-align: left;">调用</th>
      <th style="text-align: left;">理由</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>tp_submit</code></td>
      <td style="text-align: left;"><code>pthread_cond_signal</code></td>
      <td style="text-align: left;">新增<strong>一个</strong>任务只能占用<strong>一个</strong>工作线程。唤醒全部空闲线程会造成"惊群"：所有线程醒来竞争同一把锁，只有一个能拿到任务，其余重新进入等待</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tp_destroy</code></td>
      <td style="text-align: left;"><code>pthread_cond_broadcast</code></td>
      <td style="text-align: left;">关闭标志必须被<strong>每一个</strong>空闲线程观察到。若只 <code>signal</code> 一次，其余线程将永远阻塞在条件变量上，<code>pthread_join</code> 随之永久挂起</td>
    </tr>
  </tbody>
</table>


> **📌 一条通用判据**：唤醒**一个**等待者就足以处理的状态变化，用 `signal`；需要**所有**等待者都重新检查的状态变化，用 `broadcast`。


## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)

NCORE = os.cpu_count() or 1
HAS_PTHREAD = False
if CC:
    _probe = ("#include <pthread.h>\n#include <stdio.h>\n"
              "static void* f(void* a){ (void)a; return NULL; }\n"
              "int main(){ pthread_t t; pthread_create(&t,NULL,f,NULL);"
              " pthread_join(t,NULL); printf(\"ok\"); return 0; }")
    open("/tmp/_pth_probe.c", "w").write(_probe)
    _r = subprocess.run(f"{CC} -pthread /tmp/_pth_probe.c -o /tmp/_pth_probe",
                        shell=True, capture_output=True, text=True)
    HAS_PTHREAD = (_r.returncode == 0)

NT = NCORE                      # 本实验默认使用的工作线程数
print("逻辑核心:", NCORE)
print("Pthreads:", "可用" if HAS_PTHREAD else "不可用")

if CC is None:
    print("\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。")
elif not HAS_PTHREAD:
    print("\n⚠️  编译器无法链接 POSIX 线程库，请检查 -pthread 支持。")
else:
    print(f"\n✅ 环境就绪：可用 {NT} 个线程，可以开始实验！")

In [44]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(srcs, out):
    """编译一个或多个 C 源文件，返回可执行文件名；失败则打印错误。
    srcs 可以是字符串（多个文件以空格分隔）或字符串列表。
    与前几章相比唯一的变化：把 -fopenmp 换成 -pthread。"""
    if not isinstance(srcs, str):
        srcs = " ".join(srcs)
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread {srcs} -o {out} -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print("编译告警：\n", r.stderr)
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "threads", "grain (px)", "方法") \
                or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append({"method": name, "time": float(mt.group()),
                     "speedup": float(ms.group()) if ms else None})
    return rows


def parse_mpix(text):
    """取出第 4 列 Mpix/s。表格列序为 方法|耗时|加速比|Mpix/s|校验。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 4:
            continue
        name = cells[0]
        if name.lower() in ("method", "threads", "方法") or set(name) <= set("-: "):
            continue
        mg = re.search(r"[-+]?\d*\.?\d+", cells[3])
        if not mg:
            continue
        rows.append({"method": name, "mpix": float(mg.group())})
    return rows


def parse_check(text):
    """取出校验列，用于在图上标出 FAIL 的版本。"""
    st = {}
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 5:
            continue
        st[cells[0]] = cells[4]
    return st


def parse_balance(text):
    """解析 'Load balance [...]' 段落，返回 {方法名: {...}}。"""
    res, cur = {}, None
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"Load balance \[(.+?)\s*\]", s)
        if m:
            cur = m.group(1).strip()
            res[cur] = {"imb_iter": None, "imb_busy": None,
                        "idle": None, "busy": [], "tasks": []}
            continue
        if cur is None:
            continue
        if s.startswith("iterations"):
            v = re.search(r"max/avg\s*=\s*([\d.]+)", s)
            if v:
                res[cur]["imb_iter"] = float(v.group(1))
        elif s.startswith("busy time"):
            v = re.search(r"max/avg\s*=\s*([\d.]+)", s)
            i = re.search(r"idle for\s*([\d.]+)%", s)
            if v:
                res[cur]["imb_busy"] = float(v.group(1))
            if i:
                res[cur]["idle"] = float(i.group(1))
        elif s.startswith("busy_ms"):
            res[cur]["busy"] = [float(x) for x in
                                re.findall(r"\d+\.?\d*|\.\d+", s.split(":", 1)[1])]
        elif s.startswith("tasks/worker"):
            res[cur]["tasks"] = [int(x) for x in
                                 re.findall(r"\d+", s.split(":", 1)[1])]
    return res


def parse_grain(text):
    """解析任务粒度表：| 粒度 | 任务数 | 耗时 | 加速比 | 不均衡因子 | 校验 |"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 6 or not cells[0].isdigit():
            continue
        rows.append({"grain": int(cells[0]), "tasks": int(cells[1]),
                     "time": float(cells[2]),
                     "speedup": float(re.search(r"[\d.]+", cells[3]).group()),
                     "imb": float(cells[4]), "check": cells[5]})
    return rows

In [45]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title="", checks=None):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    # 校验未通过的版本一律涂成橙色并加剖面线，避免被误读成"最优"
    hatch = [None] * len(sp)
    if checks:
        for i, n in enumerate(names):
            if checks.get(n, "").upper() == "FAIL":
                colors[i] = "#E8A33D"
                hatch[i] = "//"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    for b, h in zip(bars, hatch):
        if h:
            b.set_hatch(h)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(b.get_x() + b.get_width() / 2, s, f"{s:.2f}x",
                 ha="center", va="bottom", fontsize=10)
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


def plot_mpix(rows, title="", checks=None):
    if not rows:
        print("未解析到可绘制的吞吐率。")
        return
    names = [r["method"] for r in rows]
    v = [r["mpix"] for r in rows]
    colors = ["#295E96"] * len(v)
    valid = [i for i, n in enumerate(names)
             if not (checks and checks.get(n, "").upper() == "FAIL")]
    if valid:
        colors[max(valid, key=lambda i: v[i])] = "#C7000B"
    if checks:
        for i, n in enumerate(names):
            if checks.get(n, "").upper() == "FAIL":
                colors[i] = "#E8A33D"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, v, color=colors)
    if checks:
        for b, n in zip(bars, names):
            if checks.get(n, "").upper() == "FAIL":
                b.set_hatch("//")
    for b, g in zip(bars, v):
        plt.text(b.get_x() + b.get_width() / 2, g, f"{g:.1f}",
                 ha="center", va="bottom", fontsize=10)
    plt.ylabel("Throughput (Mpix/s)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


def plot_balance(bal, title=""):
    """每个工作线程的忙碌时间，静态均分与动态队列并列对比。"""
    keys = [k for k in bal if bal[k]["busy"]]
    if not keys:
        print("未解析到负载统计。")
        return
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4),
                             squeeze=False)
    for ax, k in zip(axes[0], keys):
        b = bal[k]["busy"]
        avg = sum(b) / len(b)
        ax.bar(range(len(b)), b, color="#295E96")
        ax.axhline(avg, ls="--", c="#C7000B", lw=1.2,
                   label=f"average = {avg:.1f} ms")
        ax.set_xlabel("Worker id")
        ax.set_ylabel("Busy time (ms)")
        ax.set_title(f"{k}   max/avg = {bal[k]['imb_busy']:.2f}")
        ax.legend(fontsize=9)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_grain(rows, title=""):
    """任务粒度曲线：左轴耗时（对数横轴），右轴不均衡因子。"""
    if not rows:
        print("未解析到粒度数据。")
        return
    g = [r["grain"] for r in rows]
    t = [r["time"] for r in rows]
    imb = [r["imb"] for r in rows]
    best = t.index(min(t))
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax1.plot(g, t, "o-", c="#C7000B", lw=2, label="Time (ms)")
    ax1.plot(g[best], t[best], "*", c="#C7000B", ms=18)
    ax1.set_xscale("log", base=8)
    ax1.set_xlabel("Grain (pixels per task, log scale)")
    ax1.set_ylabel("Time (ms)")
    ax2 = ax1.twinx()
    ax2.plot(g, imb, "s--", c="#295E96", lw=1.5, label="Imbalance max/avg")
    ax2.set_ylabel("Imbalance factor")
    ax2.axhline(1.0, ls=":", c="#295E96", lw=0.8)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper center", fontsize=9)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(f'{"Grain":>10} {"Tasks":>9} {"Time(ms)":>10} {"Speedup":>9} {"Imb":>7}')
    for r in rows:
        print(f'{r["grain"]:>10} {r["tasks"]:>9} {r["time"]:>10.3f} '
              f'{r["speedup"]:>8.2f}x {r["imb"]:>7.2f}')

In [46]:
# 创建源代码目录
!mkdir -p src_tp

## 5. 线程池的实现

### 5.1 接口

接口文件只声明类型与函数，不暴露任何内部结构。这样做的好处是：**线程池的实现可以整体替换**（例如扩展实验中换成有界队列版本），而使用它的应用代码一行都不必改。


In [ ]:
%%writefile src_tp/threadpool.h
/* ==========================================================================
 * threadpool.h -- A minimal fixed-size thread pool built on POSIX threads.
 *
 * Design in one sentence: N worker threads are created once, then repeatedly
 * take tasks from one shared FIFO queue until the pool is destroyed.
 *
 * The three pieces of state that make this work:
 *   - a mutex        protecting the queue
 *   - a condition variable   telling idle workers that a task has arrived
 *   - a shutdown flag        telling workers to leave once the queue is empty
 * ========================================================================== */
#ifndef THREADPOOL_H
#define THREADPOOL_H

#include <stddef.h>

typedef struct threadpool threadpool_t;

/* A task is just a function plus one argument. The pool never looks inside
 * arg, and never frees it: ownership stays with the caller. */
typedef void (*tp_task_fn)(void *arg);

/* Create a pool with nthreads workers. Returns NULL on failure. */
threadpool_t *tp_create(int nthreads);

/* Append one task to the queue. Returns 0 on success, -1 on failure.
 * Never blocks: the queue is unbounded (see the extension lab). */
int tp_submit(threadpool_t *pool, tp_task_fn fn, void *arg);

/* Drain the queue, join every worker, release all resources.
 * DRAIN semantics: tasks already submitted are guaranteed to run. */
void tp_destroy(threadpool_t *pool);

/* Number of workers in the pool. */
int tp_thread_count(const threadpool_t *pool);

/* Index of the calling worker, 0 .. nthreads-1.
 * Returns -1 when called from a thread that is not a pool worker.
 * Tasks use this to accumulate per-thread statistics without any locking. */
int tp_worker_id(void);

#endif /* THREADPOOL_H */

### 5.2 实现

请重点阅读 `tp_worker` 函数——它是整个线程池的核心，三处关键写法均已在注释中标出。此外注意两个细节：

- **任务节点由线程池 `malloc` / `free`，任务参数 `arg` 由调用方负责**。这一约定必须明确写在接口文档中，否则必然出现重复释放或内存泄漏；
- **`tp_worker_id()` 通过 `__thread` 线程局部存储实现**。`__thread` 修饰的变量每个线程各有一份，读写它既不需要加锁，也不需要查表。任务体正是借此把统计量写入属于自己的槽位。


In [ ]:
%%writefile src_tp/threadpool.c
#include "threadpool.h"

#include <pthread.h>
#include <stdlib.h>

/* --------------------------------------------------------------------------
 * Queue node. One malloc per task; see the granularity study for the point at
 * which this cost stops being negligible.
 * -------------------------------------------------------------------------- */
typedef struct tp_task {
  tp_task_fn fn;
  void *arg;
  struct tp_task *next;
} tp_task_t;

struct threadpool {
  pthread_mutex_t lock;      /* protects head, tail and shutdown            */
  pthread_cond_t not_empty;  /* signalled when a task is appended           */
  tp_task_t *head, *tail;    /* FIFO queue                                  */
  int shutdown;              /* set by tp_destroy, read by every worker     */
  int nthreads;
  pthread_t *threads;
};

/* Thread-local worker index. Set once when the worker starts, so a running
 * task can identify itself with no lock and no lookup. */
static __thread int tp_tls_id = -1;

int tp_worker_id(void) { return tp_tls_id; }

/* --------------------------------------------------------------------------
 * The worker loop -- the heart of the pool.
 * -------------------------------------------------------------------------- */
typedef struct {
  threadpool_t *pool;
  int id;
} tp_worker_arg_t;

static void *tp_worker(void *raw) {
  tp_worker_arg_t *wa = (tp_worker_arg_t *)raw;
  threadpool_t *pool = wa->pool;
  tp_tls_id = wa->id;
  free(wa);

  for (;;) {
    pthread_mutex_lock(&pool->lock);

    /* WHILE, not IF. pthread_cond_wait may return without a task being
     * available: another worker may have taken it first, and the standard
     * also permits spurious wakeups. The predicate must be re-tested. */
    while (pool->head == NULL && !pool->shutdown)
      pthread_cond_wait(&pool->not_empty, &pool->lock);

    /* Leave only when the queue is empty AND shutdown was requested.
     * Testing shutdown first would discard tasks that were already
     * submitted -- that is the difference between "stop now" and "drain". */
    if (pool->head == NULL && pool->shutdown) {
      pthread_mutex_unlock(&pool->lock);
      break;
    }

    tp_task_t *t = pool->head;
    pool->head = t->next;
    if (pool->head == NULL) pool->tail = NULL;

    /* Release the lock BEFORE running the task. Holding it across the task
     * body would serialise the whole pool and defeat its purpose. */
    pthread_mutex_unlock(&pool->lock);

    t->fn(t->arg);
    free(t);
  }
  return NULL;
}

/* -------------------------------------------------------------------------- */
threadpool_t *tp_create(int nthreads) {
  if (nthreads <= 0) return NULL;

  threadpool_t *pool = (threadpool_t *)calloc(1, sizeof(threadpool_t));
  if (!pool) return NULL;

  pool->nthreads = nthreads;
  pool->threads = (pthread_t *)calloc((size_t)nthreads, sizeof(pthread_t));
  if (!pool->threads) {
    free(pool);
    return NULL;
  }

  pthread_mutex_init(&pool->lock, NULL);
  pthread_cond_init(&pool->not_empty, NULL);

  for (int i = 0; i < nthreads; i++) {
    tp_worker_arg_t *wa = (tp_worker_arg_t *)malloc(sizeof(tp_worker_arg_t));
    wa->pool = pool;
    wa->id = i;
    if (pthread_create(&pool->threads[i], NULL, tp_worker, wa) != 0) {
      free(wa);
      pool->nthreads = i;  /* only i workers actually started */
      tp_destroy(pool);
      return NULL;
    }
  }
  return pool;
}

int tp_submit(threadpool_t *pool, tp_task_fn fn, void *arg) {
  if (!pool || !fn) return -1;

  tp_task_t *t = (tp_task_t *)malloc(sizeof(tp_task_t));
  if (!t) return -1;
  t->fn = fn;
  t->arg = arg;
  t->next = NULL;

  pthread_mutex_lock(&pool->lock);
  if (pool->shutdown) {           /* refuse work after shutdown was requested */
    pthread_mutex_unlock(&pool->lock);
    free(t);
    return -1;
  }
  if (pool->tail) pool->tail->next = t;
  else pool->head = t;
  pool->tail = t;

  /* signal, not broadcast: one new task can only occupy one worker.
   * Waking every idle worker here would cause a thundering herd. */
  pthread_cond_signal(&pool->not_empty);
  pthread_mutex_unlock(&pool->lock);
  return 0;
}

void tp_destroy(threadpool_t *pool) {
  if (!pool) return;

  pthread_mutex_lock(&pool->lock);
  pool->shutdown = 1;
  /* broadcast, not signal: every idle worker must wake up and observe the
   * flag, otherwise some of them would block on the condition forever. */
  pthread_cond_broadcast(&pool->not_empty);
  pthread_mutex_unlock(&pool->lock);

  for (int i = 0; i < pool->nthreads; i++)
    pthread_join(pool->threads[i], NULL);

  /* Workers only exit once the queue is empty, so nothing is left to free. */
  pthread_mutex_destroy(&pool->lock);
  pthread_cond_destroy(&pool->not_empty);
  free(pool->threads);
  free(pool);
}

int tp_thread_count(const threadpool_t *pool) {
  return pool ? pool->nthreads : 0;
}

## 6. 案例任务：Mandelbrot 集渲染

### 6.1 为什么选择这个任务

线程池是本实验的主角，任务负载只是用来检验它的载荷，因此负载应当满足三个条件：**算法本身足够简单**、**结果可以严格校验**、**各任务的工作量差异显著**。Mandelbrot 集渲染同时满足这三条。

对复平面上的每一点 $c$，迭代
$$z_0 = 0,\qquad z_{n+1} = z_n^2 + c$$
直到 $|z_n| > 2$（该点必然发散）或迭代次数达到上限 `maxiter`（判定该点属于集合内部）。像素值取实际迭代次数。

由此得到三个性质：

<!--
| 性质 | 说明 | 对本实验的意义 |
|---|---|---|
| **逐像素独立** | 任一像素的计算不读取也不写入其他像素 | 任务之间无数据依赖，无需任何锁 |
| **结果完全确定** | 同一像素无论由哪个线程、以何种顺序计算，浮点运算序列完全相同 | 可用 `memcmp` 与串行结果**逐字节**比对，无需设置容差 |
| **工作量极不均衡** | 集合外部的点通常迭代几次即发散；集合内部的点必须迭代满 `maxiter` 次，二者相差可达三个数量级 | 正是暴露负载不均衡问题所需要的负载 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">性质</th>
      <th style="text-align: left;">说明</th>
      <th style="text-align: left;">对本实验的意义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>逐像素独立</strong></td>
      <td style="text-align: left;">任一像素的计算不读取也不写入其他像素</td>
      <td style="text-align: left;">任务之间无数据依赖，无需任何锁</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>结果完全确定</strong></td>
      <td style="text-align: left;">同一像素无论由哪个线程、以何种顺序计算，浮点运算序列完全相同</td>
      <td style="text-align: left;">可用 <code>memcmp</code> 与串行结果<strong>逐字节</strong>比对，无需设置容差</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>工作量极不均衡</strong></td>
      <td style="text-align: left;">集合外部的点通常迭代几次即发散；集合内部的点必须迭代满 <code>maxiter</code> 次，二者相差可达三个数量级</td>
      <td style="text-align: left;">正是暴露负载不均衡问题所需要的负载</td>
    </tr>
  </tbody>
</table>


> **📌 关于渲染窗口的选择**
>
> 代码中渲染的窗口是 $\mathrm{Re}\,c \in [-2.0, 1.0]$、$\mathrm{Im}\,c \in [-2.1, 0.9]$，**刻意不以实轴为中心**。原因是 Mandelbrot 集关于实轴对称：若采用常见的居中窗口，把图像按行**偶数等分**时上下两半的工作量会恰好相等，本实验要展示的负载不均衡现象将在最常用的线程数下消失。

### 6.2 任务的划分与四种方案

图像共 $W \times H$ 个像素，按行主序编号为 $0 \ldots WH-1$。一个任务负责一段连续的像素区间 $[p_0, p_1)$，区间长度称为**粒度**（grain）。

<!--
| 编号 | 方案 | 任务划分 | 说明 |
|---|---|---|---|
| 1 | `Serial` | 整幅图像一个区间 | 基准，同时产生用于校验的参考图像 |
| 2 | `Spawn per Task` | 与方案 4 **完全相同** | 不使用线程池：每个任务调用一次 `pthread_create`，每 $N$ 个任务一批，批内 `join` 后再开下一批 |
| 3 | `Pool + Static` | $N$ 个任务，每个覆盖 $WH/N$ 个像素 | 线程池，但任务数恰好等于线程数——相当于静态均分 |
| 4 | `Pool + Dynamic` | 按 grain 切分为大量小任务 | 线程池，工作线程按完成速度自动领取后续任务 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">编号</th>
      <th style="text-align: left;">方案</th>
      <th style="text-align: left;">任务划分</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">1</td>
      <td style="text-align: left;"><code>Serial</code></td>
      <td style="text-align: left;">整幅图像一个区间</td>
      <td style="text-align: left;">基准，同时产生用于校验的参考图像</td>
    </tr>
    <tr>
      <td style="text-align: left;">2</td>
      <td style="text-align: left;"><code>Spawn per Task</code></td>
      <td style="text-align: left;">与方案 4 <strong>完全相同</strong></td>
      <td style="text-align: left;">不使用线程池：每个任务调用一次 <code>pthread_create</code>，每 N 个任务一批，批内 <code>join</code> 后再开下一批</td>
    </tr>
    <tr>
      <td style="text-align: left;">3</td>
      <td style="text-align: left;"><code>Pool + Static</code></td>
      <td style="text-align: left;">N 个任务，每个覆盖 WH/N 个像素</td>
      <td style="text-align: left;">线程池，但任务数恰好等于线程数——相当于静态均分</td>
    </tr>
    <tr>
      <td style="text-align: left;">4</td>
      <td style="text-align: left;"><code>Pool + Dynamic</code></td>
      <td style="text-align: left;">按 grain 切分为大量小任务</td>
      <td style="text-align: left;">线程池，工作线程按完成速度自动领取后续任务</td>
    </tr>
  </tbody>
</table>


这样安排是为了得到两组**受控对照**：

- **方案 2 与方案 4** 的任务划分完全相同，唯一差别是有没有线程池 → 二者的差异即**线程创建与回收（以及批间同步）的代价**；
- **方案 3 与方案 4** 使用同一个线程池，唯一差别是任务粒度 → 二者的差异即**任务粒度对负载均衡的影响**。

程序还会为每个工作线程记录两项统计：**执行的迭代总次数**与**处于任务体内的忙碌时间**。计数器数组按 64 字节缓存行填充，以避免第四章伪共享实验中测到的那种性能损失。


In [ ]:
%%writefile src_tp/mandelbrot_pool.c
/* ==========================================================================
 * mandelbrot_pool.c -- Chapter 4 capstone benchmark.
 *
 * One workload (Mandelbrot rendering) computed four ways:
 *   1. Serial                     -- the reference result and the 1.00x base
 *   2. Spawn per Task (no pool)   -- one pthread_create/join per task
 *   3. Pool + Static              -- pool, one large chunk per worker
 *   4. Pool + Dynamic             -- pool, many small tasks in the queue
 *
 * Rows 2 and 4 use the SAME decomposition, so their difference isolates the
 * cost of creating and destroying threads.
 * Rows 3 and 4 use the SAME pool, so their difference isolates the effect of
 * task granularity on load balance.
 * ========================================================================== */
#include <pthread.h>
#include <unistd.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#include "threadpool.h"

#define NTIMES 3      // Repetitions for the parallel versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline
#define MAX_THREADS 256

// ---------------------------------------------------------
// Timing
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Per-worker statistics, padded to a full cache line.
// Without the padding the counters of neighbouring workers would share one
// cache line, and every increment would invalidate the other workers' copies
// -- the false-sharing effect measured earlier in this chapter.
typedef struct {
  unsigned long iters;  // total Mandelbrot iterations executed by this worker
  unsigned long tasks;  // number of tasks executed by this worker
  double busy_ms;       // wall time this worker spent inside task bodies
  char pad[64 - 2 * sizeof(unsigned long) - sizeof(double)];
} stat_t;

static stat_t g_stat[MAX_THREADS];

// ---------------------------------------------------------
// Image geometry. Fixed for the whole run, so a task only has to carry the
// pixel range it is responsible for.
// ---------------------------------------------------------
static int g_W, g_H, g_maxiter;

// The window rendered: real part in [-2.0, 1.0], imaginary part in
// [-2.1, 0.9].
//
// The window is deliberately NOT centred on the real axis. The Mandelbrot set
// is symmetric about that axis, so the usual centred view would make an
// even-way split of the image rows accidentally balanced, and the load
// imbalance this experiment is built to demonstrate would disappear for
// exactly the thread counts most likely to be used.
#define VIEW_X0 (-2.0)
#define VIEW_Y0 (-2.1)
#define VIEW_SPAN (3.0)

// ---------------------------------------------------------
// The kernel: render the pixels [p0, p1) of the image, in row-major order.
// Returns the number of iterations actually executed, which is the true
// measure of the work done -- pixel count is not, and that is exactly why
// this workload is load-imbalanced.
// ---------------------------------------------------------
static unsigned long mandel_range(uint16_t *img, long p0, long p1) {
  const double dx = VIEW_SPAN / (double)g_W;
  const double dy = VIEW_SPAN / (double)g_H;
  unsigned long iters = 0;

  for (long p = p0; p < p1; p++) {
    int py = (int)(p / g_W);
    int px = (int)(p % g_W);
    double cr = VIEW_X0 + ((double)px + 0.5) * dx;
    double ci = VIEW_Y0 + ((double)py + 0.5) * dy;
    double zr = 0.0, zi = 0.0;
    int it = 0;
    while (it < g_maxiter) {
      double zr2 = zr * zr, zi2 = zi * zi;
      if (zr2 + zi2 > 4.0) break;   // escaped: this pixel is finished early
      zi = 2.0 * zr * zi + ci;
      zr = zr2 - zi2 + cr;
      it++;
    }
    img[p] = (uint16_t)it;
    iters += (unsigned long)it;
  }
  return iters;
}

// ---------------------------------------------------------
// Task descriptor. The pool never looks inside it and never frees it:
// the whole array lives in main and outlives the pool.
// ---------------------------------------------------------
typedef struct {
  uint16_t *img;
  long p0, p1;
} range_t;

// Task entry point for the pool. tp_worker_id() lets the task find its own
// slot in g_stat, so the statistics need no lock at all.
static void mandel_task(void *arg) {
  range_t *r = (range_t *)arg;
  double t0 = get_time_ms();
  unsigned long it = mandel_range(r->img, r->p0, r->p1);
  double t1 = get_time_ms();
  int id = tp_worker_id();
  if (id >= 0 && id < MAX_THREADS) {
    g_stat[id].iters += it;
    g_stat[id].tasks += 1;
    g_stat[id].busy_ms += t1 - t0;
  }
}

// Thread entry point for the no-pool version. Same work, different plumbing.
static void *mandel_thread(void *arg) {
  range_t *r = (range_t *)arg;
  mandel_range(r->img, r->p0, r->p1);
  return NULL;
}

// ---------------------------------------------------------
// Build a task list that splits [0, npix) into chunks of at most grain pixels
// Returns the number of tasks written into tasks[].
// ---------------------------------------------------------
static long build_tasks(range_t *tasks, long ntask_max, uint16_t *img,
                        long npix, long grain) {
  long n = 0;
  for (long p = 0; p < npix && n < ntask_max; p += grain) {
    tasks[n].img = img;
    tasks[n].p0 = p;
    tasks[n].p1 = (p + grain < npix) ? p + grain : npix;
    n++;
  }
  return n;
}

// ---------------------------------------------------------
// Method 2: no pool. One thread per task, launched in waves of nthreads.
// A wave has to be joined before the next one starts, because without a
// persistent set of workers there is no other way to bound concurrency.
// ---------------------------------------------------------
static void run_spawn(range_t *tasks, long ntasks, int nthreads) {
  pthread_t th[MAX_THREADS];
  long i = 0;
  while (i < ntasks) {
    int n = 0;
    while (n < nthreads && i < ntasks) {
      if (pthread_create(&th[n], NULL, mandel_thread, &tasks[i]) != 0) break;
      n++;
      i++;
    }
    for (int k = 0; k < n; k++) pthread_join(th[k], NULL);
  }
}

// ---------------------------------------------------------
// Methods 3 and 4: the pool. Identical code -- only the task list differs.
//
// tp_destroy() is what makes this correct: it drains the queue and joins
// every worker, so on return all tasks are guaranteed to have finished.
// The base pool has no separate "wait for completion" call, which is why the
// pool has to be created and destroyed once per measured run. Adding that
// call is the extension exercise.
// ---------------------------------------------------------
static void run_pool(range_t *tasks, long ntasks, int nthreads) {
  threadpool_t *pool = tp_create(nthreads);
  if (!pool) return;
  for (long i = 0; i < ntasks; i++) tp_submit(pool, mandel_task, &tasks[i]);
  tp_destroy(pool);
}

static void stat_reset(void) { memset(g_stat, 0, sizeof(g_stat)); }

// Report how evenly the work was distributed, using two independent measures.
//
//   iterations : how much arithmetic each worker performed. For the static
//                split this is a property of the decomposition alone, so it
//                is reproducible on any machine.
//   busy time  : how long each worker actually spent inside task bodies. The
//                parallel run cannot finish before the slowest worker does,
//                so 1 - avg/max is the fraction of the parallel time during
//                which the average worker had nothing left to do.
//
// The imbalance factor max/avg bounds the achievable speedup from above:
// speedup <= nthreads / (max/avg).
static void report_balance(const char *name, int nthreads) {
  unsigned long tot_i = 0, mx_i = 0, mn_i = (unsigned long)-1;
  double tot_b = 0.0, mx_b = 0.0;
  for (int i = 0; i < nthreads; i++) {
    unsigned long v = g_stat[i].iters;
    tot_i += v;
    if (v > mx_i) mx_i = v;
    if (v < mn_i) mn_i = v;
    tot_b += g_stat[i].busy_ms;
    if (g_stat[i].busy_ms > mx_b) mx_b = g_stat[i].busy_ms;
  }
  if (tot_i == 0) return;
  double avg_i = (double)tot_i / (double)nthreads;
  double avg_b = tot_b / (double)nthreads;
  printf("  Load balance [%s]\n", name);
  printf("    iterations : max/avg = %.2f   (least loaded worker %.1f%%, "
         "most loaded %.1f%% of all work)\n",
         (double)mx_i / avg_i, 100.0 * (double)mn_i / (double)tot_i,
         100.0 * (double)mx_i / (double)tot_i);
  printf("    busy time  : max/avg = %.2f   (average worker idle for %.1f%% "
         "of the parallel run)\n",
         mx_b > 0.0 ? mx_b / avg_b : 0.0,
         mx_b > 0.0 ? 100.0 * (1.0 - avg_b / mx_b) : 0.0);
  printf("    busy_ms    :");
  for (int i = 0; i < nthreads && i < 64; i++) printf(" %.1f", g_stat[i].busy_ms);
  if (nthreads > 64) printf(" ...");
  printf("\n");
  printf("    tasks/worker:");
  for (int i = 0; i < nthreads && i < 64; i++) printf(" %lu", g_stat[i].tasks);
  if (nthreads > 64) printf(" ...");
  printf("\n");
}

// Print one row of the report table
static void report(const char *name, double t, double t_base, double npix,
                   const char *chk) {
  printf("| %-20s | %9.3f | %5.2f x | %7.2f | %5s |\n", name, t, t_base / t,
         npix / (t * 1000.0), chk);
}

// Byte-exact comparison. Every version executes exactly the same arithmetic
// in the same order for a given pixel, so anything other than an identical
// image is a defect -- no tolerance is needed here.
static const char *check_image(const uint16_t *ref, const uint16_t *test,
                               long npix) {
  return memcmp(ref, test, (size_t)npix * sizeof(uint16_t)) == 0 ? "PASS"
                                                                 : "FAIL";
}

static void write_pgm(const char *path, const uint16_t *img, int W, int H,
                      int maxiter) {
  FILE *f = fopen(path, "wb");
  if (!f) return;
  fprintf(f, "P5\n%d %d\n255\n", W, H);
  for (long p = 0; p < (long)W * H; p++) {
    int v = (img[p] >= maxiter) ? 0 : (int)(255.0 * img[p] / maxiter);
    fputc(v, f);
  }
  fclose(f);
  printf("Image written to %s\n", path);
}

// ---------------------------------------------------------
int main(int argc, char **argv) {
  if (argc < 4) {
    printf("Usage: %s <W> <H> <maxiter> [threads] [grain] [out.pgm]\n", argv[0]);
    printf("Example: %s 1024 1024 2000 8 1024\n", argv[0]);
    printf("  grain = pixels per task (default: W, i.e. one image row)\n");
    return 1;
  }

  g_W = atoi(argv[1]);
  g_H = atoi(argv[2]);
  g_maxiter = atoi(argv[3]);
  int threads = (argc > 4) ? atoi(argv[4]) : 0;
  long grain = (argc > 5) ? atol(argv[5]) : 0;
  const char *pgm = (argc > 6) ? argv[6] : NULL;

  if (g_W <= 0 || g_H <= 0 || g_maxiter <= 0) return 1;
  if (g_maxiter > 65535) g_maxiter = 65535;   // the image stores uint16 counts
  if (threads <= 0) threads = (int)sysconf(_SC_NPROCESSORS_ONLN);
  if (threads > MAX_THREADS) threads = MAX_THREADS;
  if (grain <= 0) grain = g_W;

  long npix = (long)g_W * (long)g_H;
  long ntasks_dyn = (npix + grain - 1) / grain;
  long grain_static = (npix + threads - 1) / threads;

  printf("============================================================\n");
  printf(" Chapter 4 capstone: a Pthreads thread pool on Mandelbrot\n");
  printf(" Image  : %d x %d  (%.2f Mpix), max iterations %d\n", g_W, g_H,
         (double)npix / 1e6, g_maxiter);
  printf(" Threads: %d   Grain: %ld pixels/task  ->  %ld tasks\n", threads,
         grain, ntasks_dyn);
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  size_t bytes = (size_t)npix * sizeof(uint16_t);
  uint16_t *img_ref = (uint16_t *)malloc(bytes);
  uint16_t *img = (uint16_t *)malloc(bytes);
  range_t *tasks = (range_t *)malloc(sizeof(range_t) * (size_t)(ntasks_dyn + threads));
  if (!img_ref || !img || !tasks) {
    printf("Alloc failed\n");
    return 1;
  }

  // Golden reference, produced serially
  memset(img_ref, 0, bytes);
  mandel_range(img_ref, 0, npix);

  double start, end;

  printf("\n----------------------------------------------------------------\n");
  printf("| %-20s | %9s | %7s | %7s | %5s |\n", "Method", "Time (ms)",
         "Speedup", "Mpix/s", "Check");
  printf("|----------------------|-----------|---------|---------|-------|\n");

  // 1. Serial
  memset(img, 0, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) mandel_range(img, 0, npix);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial", t_base, t_base, (double)npix, check_image(img_ref, img, npix));

  // 2. No pool: one pthread_create per task, same task list as method 4
  long nt = build_tasks(tasks, ntasks_dyn + threads, img, npix, grain);
  memset(img, 0, bytes);   // clear: a stale correct image would mask an error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) run_spawn(tasks, nt, threads);
  end = get_time_ms();
  double t_spawn = (end - start) / NTIMES;
  report("Spawn per Task", t_spawn, t_base, (double)npix,
         check_image(img_ref, img, npix));

  // 3. Pool, static split: exactly one chunk per worker
  nt = build_tasks(tasks, ntasks_dyn + threads, img, npix, grain_static);
  memset(img, 0, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) {
    // Collect the statistics of the LAST repetition only. Summing them over
    // several runs would blur the picture, because which worker picks up
    // which task may differ from run to run.
    if (t == NTIMES - 1) stat_reset();
    run_pool(tasks, nt, threads);
  }
  end = get_time_ms();
  double t_static = (end - start) / NTIMES;
  report("Pool + Static", t_static, t_base, (double)npix,
         check_image(img_ref, img, npix));
  static stat_t saved[MAX_THREADS];
  memcpy(saved, g_stat, sizeof(saved));

  // 4. Pool, dynamic queue: many small tasks, workers take the next one
  nt = build_tasks(tasks, ntasks_dyn + threads, img, npix, grain);
  memset(img, 0, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) {
    if (t == NTIMES - 1) stat_reset();
    run_pool(tasks, nt, threads);
  }
  end = get_time_ms();
  double t_dyn = (end - start) / NTIMES;
  report("Pool + Dynamic", t_dyn, t_base, (double)npix,
         check_image(img_ref, img, npix));

  printf("----------------------------------------------------------------\n");

  // Load balance of methods 3 and 4, from the per-worker iteration counters
  static stat_t cur[MAX_THREADS];
  memcpy(cur, g_stat, sizeof(cur));
  memcpy(g_stat, saved, sizeof(g_stat));
  report_balance("Pool + Static ", threads);
  memcpy(g_stat, cur, sizeof(g_stat));
  report_balance("Pool + Dynamic", threads);

  if (pgm) write_pgm(pgm, img_ref, g_W, g_H, g_maxiter);

  free(img_ref);
  free(img);
  free(tasks);
  return 0;
}

In [ ]:
BIN = compile_c("src_tp/mandelbrot_pool.c src_tp/threadpool.c",
                "src_tp/mandelbrot_pool")
# 参数：宽 高 最大迭代次数 线程数 粒度(像素/任务) [输出图像]
out_mb = run_bin(BIN, 1024, 1024, 2000, NT, 1024, "src_tp/mandelbrot.pgm")

In [ ]:
# 查看渲染结果，确认计算的确是一幅 Mandelbrot 图像
import numpy as np, matplotlib.pyplot as plt

def read_pgm(path):
    with open(path, "rb") as f:
        assert f.readline().strip() == b"P5"
        w, h = map(int, f.readline().split())
        int(f.readline())
        return np.frombuffer(f.read(w * h), dtype=np.uint8).reshape(h, w)

img = read_pgm("src_tp/mandelbrot.pgm")
plt.figure(figsize=(5.5, 5.5))
plt.imshow(img, cmap="magma", origin="lower")
plt.title("Mandelbrot set (rendering window used in this lab)")
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. 结果分析（一）：为什么需要池，为什么需要队列

请对照上表的四行数据回答两个问题。

### 7.1 方案 2 与方案 4：线程池省下了什么

两行的任务划分完全一致，因此差异只可能来自两处：

1. **线程创建与回收的次数**。方案 4 全程只创建 $N$ 个线程；方案 2 创建的线程数等于任务数。当任务数为数千时，这一项从"一次性开销"变成了"主要开销"；
2. **批间同步**。方案 2 每批必须 `join` 完才能开下一批，于是每一批的耗时都由该批中最慢的线程决定，$K$ 批就累积 $K$ 次这样的等待。线程池没有批的概念，快的线程直接领取下一个任务。

第二点常被忽略，但它恰恰说明：线程池的价值**不只是省去创建开销**，更在于它取消了人为的同步点。

### 7.2 方案 3 与方案 4：静态均分的问题

两行使用同一个线程池、同一份代码，唯一差别是任务粒度：方案 3 把图像切成 $N$ 块（每线程恰好一块），方案 4 切成数千块。

若各像素的计算量相同，两者应当没有区别。但 Mandelbrot 的计算量与像素位置强相关——集合内部的像素要迭代满 `maxiter` 次，外部的像素往往几次就结束。于是按行等分时，落在集合稠密区的那一块工作量可能是空白区那一块的十几倍。**并行执行的总时间由最慢的线程决定**，其余线程完成后只能空等。

下一节用实测的每线程统计定量说明这一点。


In [ ]:
rows_mb = parse_table(out_mb)
chk_mb = parse_check(out_mb)
plot_speedup(rows_mb, "Mandelbrot: four execution strategies", chk_mb)
plot_mpix(parse_mpix(out_mb), "Throughput (higher is better)", chk_mb)

## 8. 负载均衡的量化

程序输出的 `Load balance` 段落给出了两套独立的度量：

<!--
| 度量 | 含义 | 特点 |
|---|---|---|
| **迭代次数** | 该线程执行的 Mandelbrot 迭代总数 | 对静态均分而言，它只由划分方式决定，与机器无关，因此可重复 |
| **忙碌时间** | 该线程处于任务体内部的累计墙钟时间 | 直接对应"谁拖住了整体进度" |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">度量</th>
      <th style="text-align: left;">含义</th>
      <th style="text-align: left;">特点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>迭代次数</strong></td>
      <td style="text-align: left;">该线程执行的 Mandelbrot 迭代总数</td>
      <td style="text-align: left;">对静态均分而言，它只由划分方式决定，与机器无关，因此可重复</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>忙碌时间</strong></td>
      <td style="text-align: left;">该线程处于任务体内部的累计墙钟时间</td>
      <td style="text-align: left;">直接对应"谁拖住了整体进度"</td>
    </tr>
  </tbody>
</table>


刻画分布的指标取**不均衡因子**
$$\text{imbalance} = \frac{\max_i W_i}{\frac{1}{N}\sum_i W_i}$$

其中 $W_i$ 为第 $i$ 个线程承担的工作量。该指标的意义十分直接：并行执行的时间不会短于最慢线程的时间，因此

$$S \le \frac{N}{\text{imbalance}}$$

即**不均衡因子直接压低了可达加速比的上限**。理想情况为 $1.00$。同时输出的"平均线程空闲比例" $1 - \overline{W}/\max W$ 表示：在整个并行执行期间，平均而言有多大比例的时间线程无事可做。

请把这两个数字与第 7 节表格中方案 3、方案 4 的加速比对照：**静态均分的加速比应当与 $N/\text{imbalance}$ 接近**，若明显更低，则说明除负载不均衡外还有其他损耗（例如内存带宽或 NUMA 远程访问）。


In [ ]:
bal = parse_balance(out_mb)
for k, v in bal.items():
    print(f"{k:16s}  迭代不均衡 = {v['imb_iter']:.2f}   "
          f"忙时不均衡 = {v['imb_busy']:.2f}   "
          f"平均空闲 = {v['idle']:.1f}%   "
          f"加速比上限 ≈ {NT / v['imb_busy']:.1f}x")
plot_balance(bal, "Busy time per worker (lower spread is better)")

## 9. 任务粒度研究

第 8 节说明"任务越小、负载越均衡"。若把这一结论推到极限——每个像素一个任务——性能是否最好？

答案是否定的，因为每个任务都要付出固定成本：

- 一次 `malloc` 与一次 `free`（任务节点）；
- 提交时一次加锁—解锁，取出时又一次加锁—解锁；
- 而且**所有线程共用同一把锁**，线程数越多，这把锁的竞争越激烈。

于是总时间可以粗略写成两项之和：

$$T(g) \;\approx\; \underbrace{\frac{W}{N}\cdot \text{imbalance}(g)}_{\text{负载不均衡，随 } g \text{ 增大而增大}} \;+\; \underbrace{\frac{WH}{g}\cdot c_{\text{task}}}_{\text{每任务固定开销，随 } g \text{ 增大而减小}}$$

两项的变化方向相反，因此 $T(g)$ 是一条**下凹曲线**，存在一个最优粒度。本节固定线程数、固定图像，只扫描粒度 $g$，把这条曲线测出来。

> **⚙️ 工程经验**：任务数取线程数的 $10$–$100$ 倍通常是合理的起点——既足以让动态领取抹平负载差异，又不至于让队列开销显现。但最优值依赖于单个任务的时长，必须实测确定。


In [ ]:
%%writefile src_tp/tp_grain.c
/* ==========================================================================
 * tp_grain.c -- Task granularity study.
 *
 * The pool, the workload and the thread count are all held fixed; the only
 * thing that changes is how many pixels one task covers.
 *
 *   fine grain  -> many small tasks -> good load balance,
 *                                      but one malloc + two mutex operations
 *                                      per task, all on ONE shared lock
 *   coarse grain-> few large tasks  -> negligible queue overhead,
 *                                      but the last worker to finish decides
 *                                      the total time
 *
 * The optimum is the grain at which those two costs are jointly smallest.
 * The Mandelbrot kernel below is identical to the one in mandelbrot_pool.c.
 * ========================================================================== */
#include <pthread.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <unistd.h>

#include "threadpool.h"

#define NTIMES 3
#define MAX_THREADS 256

#define VIEW_X0 (-2.0)
#define VIEW_Y0 (-2.1)
#define VIEW_SPAN (3.0)

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

typedef struct {
  double busy_ms;
  char pad[64 - sizeof(double)];
} stat_t;

static stat_t g_stat[MAX_THREADS];
static int g_W, g_H, g_maxiter;

static void mandel_range(uint16_t *img, long p0, long p1) {
  const double dx = VIEW_SPAN / (double)g_W;
  const double dy = VIEW_SPAN / (double)g_H;
  for (long p = p0; p < p1; p++) {
    int py = (int)(p / g_W);
    int px = (int)(p % g_W);
    double cr = VIEW_X0 + ((double)px + 0.5) * dx;
    double ci = VIEW_Y0 + ((double)py + 0.5) * dy;
    double zr = 0.0, zi = 0.0;
    int it = 0;
    while (it < g_maxiter) {
      double zr2 = zr * zr, zi2 = zi * zi;
      if (zr2 + zi2 > 4.0) break;
      zi = 2.0 * zr * zi + ci;
      zr = zr2 - zi2 + cr;
      it++;
    }
    img[p] = (uint16_t)it;
  }
}

typedef struct {
  uint16_t *img;
  long p0, p1;
} range_t;

static void mandel_task(void *arg) {
  range_t *r = (range_t *)arg;
  double t0 = get_time_ms();
  mandel_range(r->img, r->p0, r->p1);
  double t1 = get_time_ms();
  int id = tp_worker_id();
  if (id >= 0 && id < MAX_THREADS) g_stat[id].busy_ms += t1 - t0;
}

// Imbalance factor max/avg over the workers' busy time, from one run
static double imbalance(int nthreads) {
  double tot = 0.0, mx = 0.0;
  for (int i = 0; i < nthreads; i++) {
    tot += g_stat[i].busy_ms;
    if (g_stat[i].busy_ms > mx) mx = g_stat[i].busy_ms;
  }
  if (tot <= 0.0) return 0.0;
  return mx / (tot / (double)nthreads);
}

int main(int argc, char **argv) {
  if (argc < 4) {
    printf("Usage: %s <W> <H> <maxiter> [threads]\n", argv[0]);
    return 1;
  }
  g_W = atoi(argv[1]);
  g_H = atoi(argv[2]);
  g_maxiter = atoi(argv[3]);
  int threads = (argc > 4) ? atoi(argv[4]) : 0;
  if (g_W <= 0 || g_H <= 0 || g_maxiter <= 0) return 1;
  if (g_maxiter > 65535) g_maxiter = 65535;
  if (threads <= 0) threads = (int)sysconf(_SC_NPROCESSORS_ONLN);
  if (threads > MAX_THREADS) threads = MAX_THREADS;

  long npix = (long)g_W * (long)g_H;
  long grain_static = (npix + threads - 1) / threads;

  printf("============================================================\n");
  printf(" Task granularity study (thread pool, fixed thread count)\n");
  printf(" Image  : %d x %d  (%.2f Mpix), max iterations %d\n", g_W, g_H,
         (double)npix / 1e6, g_maxiter);
  printf(" Threads: %d   Repetitions: %d\n", threads, NTIMES);
  printf(" The largest grain listed equals npix/threads, i.e. the static split\n");
  printf("============================================================\n");

  size_t bytes = (size_t)npix * sizeof(uint16_t);
  uint16_t *img_ref = (uint16_t *)malloc(bytes);
  uint16_t *img = (uint16_t *)malloc(bytes);
  if (!img_ref || !img) return 1;

  memset(img_ref, 0, bytes);
  double s0 = get_time_ms();
  mandel_range(img_ref, 0, npix);
  double t_serial = get_time_ms() - s0;

  // Candidate grains, from very fine to the static split
  long cand[16];
  int nc = 0;
  for (long g = 1; g <= npix && nc < 14; g *= 8) cand[nc++] = g;
  cand[nc++] = grain_static;
  // sort ascending, then drop duplicates, so the table reads fine -> coarse
  for (int a = 0; a < nc; a++)
    for (int b = a + 1; b < nc; b++)
      if (cand[b] < cand[a]) {
        long tmp = cand[a];
        cand[a] = cand[b];
        cand[b] = tmp;
      }
  int nu = 0;
  for (int a = 0; a < nc; a++)
    if (a == 0 || cand[a] != cand[a - 1]) cand[nu++] = cand[a];
  nc = nu;

  range_t *tasks = (range_t *)malloc(sizeof(range_t) * (size_t)(npix + threads));
  if (!tasks) return 1;

  printf("\n-----------------------------------------------------------------------\n");
  printf("| %10s | %9s | %9s | %7s | %9s | %5s |\n", "Grain (px)", "Tasks",
         "Time (ms)", "Speedup", "Imbalance", "Check");
  printf("|------------|-----------|-----------|---------|-----------|-------|\n");
  printf("| %10s | %9d | %9.3f | %5.2f x | %9s | %5s |\n", "serial", 1,
         t_serial, 1.0, "-", "-");

  double best_t = 1e30;
  long best_g = 0;
  for (int c = 0; c < nc; c++) {
    long grain = cand[c];
    long nt = 0;
    for (long p = 0; p < npix; p += grain) {
      tasks[nt].img = img;
      tasks[nt].p0 = p;
      tasks[nt].p1 = (p + grain < npix) ? p + grain : npix;
      nt++;
    }

    memset(img, 0, bytes);
    double start = get_time_ms();
    for (int t = 0; t < NTIMES; t++) {
      if (t == NTIMES - 1) memset(g_stat, 0, sizeof(g_stat));
      threadpool_t *pool = tp_create(threads);
      if (!pool) return 1;
      for (long i = 0; i < nt; i++) tp_submit(pool, mandel_task, &tasks[i]);
      tp_destroy(pool);   /* drain = the completion barrier */
    }
    double t_par = (get_time_ms() - start) / NTIMES;
    const char *chk =
        memcmp(img_ref, img, bytes) == 0 ? "PASS" : "FAIL";

    printf("| %10ld | %9ld | %9.3f | %5.2f x | %9.2f | %5s |\n", grain, nt,
           t_par, t_serial / t_par, imbalance(threads), chk);
    if (t_par < best_t) {
      best_t = t_par;
      best_g = grain;
    }
  }
  printf("-----------------------------------------------------------------------\n");
  printf("  Best grain: %ld pixels/task  (%.2f x over serial, %ld tasks)\n",
         best_g, t_serial / best_t, (npix + best_g - 1) / best_g);

  free(img_ref);
  free(img);
  free(tasks);
  return 0;
}

In [ ]:
BIN = compile_c("src_tp/tp_grain.c src_tp/threadpool.c", "src_tp/tp_grain")
# 参数：宽 高 最大迭代次数 线程数
out_grain = run_bin(BIN, 1024, 1024, 2000, NT)

In [ ]:
rows_g = parse_grain(out_grain)
plot_grain(rows_g, f"Task granularity sweep ({NT} threads)")

## 10. 故障注入：两处一行之差

第 3 节论述的两条规则，本节用实验验证。做法是把 `threadpool.c` 复制一份、只改动一行，其余完全不变。

<!--
| 注入 | 改动 | 预期现象 |
|---|---|---|
| **注入一** | `while (head == NULL && !shutdown)` → `if (...)` | 工作线程在队列为空时继续向下执行，对空指针解引用，**进程被信号终止**（段错误） |
| **注入二** | `if (head == NULL && shutdown)` → `if (shutdown)` | 关闭语义由 drain 变为 stop-now，队列中剩余任务被丢弃，图像不完整，**校验报 `FAIL`**，但程序正常退出、无任何报错 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">注入</th>
      <th style="text-align: left;">改动</th>
      <th style="text-align: left;">预期现象</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>注入一</strong></td>
      <td style="text-align: left;"><code>while (head == NULL && !shutdown)</code> → <code>if (...)</code></td>
      <td style="text-align: left;">工作线程在队列为空时继续向下执行，对空指针解引用，<strong>进程被信号终止</strong>（段错误）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>注入二</strong></td>
      <td style="text-align: left;"><code>if (head == NULL && shutdown)</code> → <code>if (shutdown)</code></td>
      <td style="text-align: left;">关闭语义由 drain 变为 stop-now，队列中剩余任务被丢弃，图像不完整，<strong>校验报 <code>FAIL</code></strong>，但程序正常退出、无任何报错</td>
    </tr>
  </tbody>
</table>


> **⚠️ 请注意二者失效方式的差别。** 注入一是**显式失败**：程序崩溃，问题立刻暴露。注入二是**静默失败**：退出码为 0、没有警告、没有崩溃，只有结果是错的。并行程序中真正危险的正是后者——这也是本课程反复强调"每个版本都必须有可执行的正确性校验"的原因。


In [ ]:
import subprocess, os

base = open("src_tp/threadpool.c").read()

# 注入一：while → if
bug1 = base.replace("    while (pool->head == NULL && !pool->shutdown)",
                    "    if (pool->head == NULL && !pool->shutdown)")
assert bug1 != base, "注入一：未找到目标行，请检查源文件是否被修改过"
open("src_tp/threadpool_bug1.c", "w").write(bug1)

# 注入二：drain → stop-now
bug2 = base.replace("    if (pool->head == NULL && pool->shutdown) {",
                    "    if (pool->shutdown) {")
assert bug2 != base, "注入二：未找到目标行，请检查源文件是否被修改过"
open("src_tp/threadpool_bug2.c", "w").write(bug2)

print("已生成两个故障注入版本。")

In [ ]:
def run_capture(binary, *args, timeout=600):
    """运行程序并返回 (返回码, 标准输出)。故障版本可能被信号终止，
    因此这里不能沿用 run_bin —— 必须显式检查返回码。"""
    r = subprocess.run([f"./{binary}"] + [str(a) for a in args],
                       capture_output=True, text=True, timeout=timeout)
    return r.returncode, r.stdout


# ---- 注入一：条件等待误用 if ----
B1 = compile_c("src_tp/mandelbrot_pool.c src_tp/threadpool_bug1.c",
               "src_tp/mb_bug1")

# 竞态的触发依赖调度时序，粒度越细、线程越多越容易命中。
# 这里按由易到难的顺序尝试若干组参数，命中即停止。
rc, out1, used = 0, "", None
for grain, th in [(1, NT), (1, max(NT, 8)), (4, max(NT, 8)), (16, max(NT, 16))]:
    rc, out1 = run_capture(B1, 256, 256, 200, th, grain)
    used = (grain, th)
    if rc != 0:
        break

print(out1[-600:] if out1 else "(进程在打印任何结果之前即已终止)")
print(f"\n参数：粒度 = {used[0]} 像素/任务，线程数 = {used[1]}")
if rc < 0:
    sig = -rc
    name = "SIGSEGV 段错误" if sig == 11 else f"信号 {sig}，见 signal(7)"
    print(f"💥 注入一：进程被信号终止（{name}）。")
    print("   与第 3.1 节的分析一致：pthread_cond_wait 返回并不意味着队列非空，")
    print("   写成 if 时线程会直接对空指针 pool->head 解引用。")
elif rc != 0:
    print(f"💥 注入一：进程异常退出，返回码 {rc}。")
else:
    print("⚠️ 本次未触发崩溃。该缺陷是竞态，不必然复现，")
    print("   请重复执行本单元格，或进一步增大线程数后重试。")
    print("   请注意：**未复现不等于代码正确**——这正是竞态缺陷难以排查的原因。")

In [ ]:
# ---- 注入二：关闭语义写反 ----
B2 = compile_c("src_tp/mandelbrot_pool.c src_tp/threadpool_bug2.c",
               "src_tp/mb_bug2")
rc, out2 = run_capture(B2, 384, 384, 500, NT, 384)
print(out2)
chk2 = parse_check(out2)
bad = [k for k, v in chk2.items() if v.upper() == "FAIL"]
print(f"返回码 = {rc}（0 表示程序自认为正常结束）")
print(f"校验未通过的方案：{bad if bad else '无'}")
print("\n注意 `Spawn per Task` 一行仍为 PASS —— 它根本不使用线程池，"
      "因此不受这处改动影响。这正是受控对照的价值：\n"
      "同一次运行中，受影响与不受影响的方案同时出现，故障范围一目了然。")

## 11. 结果分析（二）：结论汇总

请基于**本次运行**的数据填写下表，并逐条给出解释。

<!--
| 观察项 | 数据来源 | 应当得出的结论 |
|---|---|---|
| `Spawn per Task` 与 `Pool + Dynamic` 的耗时之比 | 第 6 节表格第 2、4 行 | 线程创建回收与批间同步的总代价。任务越多、单任务越短，该比值越大 |
| `Pool + Static` 的不均衡因子 | 第 8 节输出 | 按行等分对本负载而言不适用；加速比上限约为 $N/	ext{imbalance}$ |
| `Pool + Dynamic` 的不均衡因子 | 第 8 节输出 | 应接近 $1.0$。工作线程按完成速度领取任务，本身就是一种动态负载均衡 |
| 最优粒度及其任务数 | 第 9 节曲线 | 任务数通常在线程数的 $10$–$100$ 倍之间；两侧的性能下降分别由队列开销与负载不均衡主导 |
| 粒度为 1 时的耗时 | 第 9 节表格首行 | 每任务固定开销完全主导。若该行甚至慢于串行，说明锁竞争已成为瓶颈 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">观察项</th>
      <th style="text-align: left;">数据来源</th>
      <th style="text-align: left;">应当得出的结论</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>Spawn per Task</code> 与 <code>Pool + Dynamic</code> 的耗时之比</td>
      <td style="text-align: left;">第 6 节表格第 2、4 行</td>
      <td style="text-align: left;">线程创建回收与批间同步的总代价。任务越多、单任务越短，该比值越大</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Pool + Static</code> 的不均衡因子</td>
      <td style="text-align: left;">第 8 节输出</td>
      <td style="text-align: left;">按行等分对本负载而言不适用；加速比上限约为 <em>N</em> / imbalance</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Pool + Dynamic</code> 的不均衡因子</td>
      <td style="text-align: left;">第 8 节输出</td>
      <td style="text-align: left;">应接近 1.0。工作线程按完成速度领取任务，本身就是一种动态负载均衡</td>
    </tr>
    <tr>
      <td style="text-align: left;">最优粒度及其任务数</td>
      <td style="text-align: left;">第 9 节曲线</td>
      <td style="text-align: left;">任务数通常在线程数的 10–100 倍之间；两侧的性能下降分别由队列开销与负载不均衡主导</td>
    </tr>
    <tr>
      <td style="text-align: left;">粒度为 1 时的耗时</td>
      <td style="text-align: left;">第 9 节表格首行</td>
      <td style="text-align: left;">每任务固定开销完全主导。若该行甚至慢于串行，说明锁竞争已成为瓶颈</td>
    </tr>
  </tbody>
</table>


### 需要提醒的两点

1. **加速比的上限不是线程数，而是 $N/\text{imbalance}$。** 在讨论"为什么没有达到线性加速"之前，应先测量不均衡因子——若它是 $3.3$，则加速比不可能超过 $N/3.3$，此时再去排查内存带宽是没有意义的。
2. **本实验的任务之间没有任何共享写入**，因此不存在数据竞争；线程池内部的共享结构只有任务队列一处，且已由互斥锁完整保护。若任务体本身需要写共享数据，则必须另行同步——线程池并不会替应用解决这一问题。


## 12. 🚀 扩展实验：有界队列与完成栅栏

基础版线程池有两处明显不足，本节请你补全。

<!--
| 不足 | 后果 | 本节的解决方案 |
|---|---|---|
| **没有"等待全部完成"的接口** | 唯一的完成保证来自 `tp_destroy`，因此每渲染一幅图像就得重建一次线程池，线程创建开销又回来了 | 增加 `tp2_wait()`：返回时保证此前提交的任务全部执行完毕，且线程池仍可继续使用 |
| **队列无上限** | 生产者比工作线程快时队列无限增长，内存占用不可控。这在流式场景（不断到达的请求、逐帧解码）中是真实的风险 | 增加容量参数：队列满时 `tp2_submit()` **阻塞**，即生产者—消费者中的**背压（backpressure）** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">不足</th>
      <th style="text-align: left;">后果</th>
      <th style="text-align: left;">本节的解决方案</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>没有"等待全部完成"的接口</strong></td>
      <td style="text-align: left;">唯一的完成保证来自 <code>tp_destroy</code>，因此每渲染一幅图像就得重建一次线程池，线程创建开销又回来了</td>
      <td style="text-align: left;">增加 <code>tp2_wait()</code>：返回时保证此前提交的任务全部执行完毕，且线程池仍可继续使用</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>队列无上限</strong></td>
      <td style="text-align: left;">生产者比工作线程快时队列无限增长，内存占用不可控。这在流式场景（不断到达的请求、逐帧解码）中是真实的风险</td>
      <td style="text-align: left;">增加容量参数：队列满时 <code>tp2_submit()</code> <strong>阻塞</strong>，即生产者—消费者中的<strong>背压（backpressure）</strong></td>
    </tr>
  </tbody>
</table>


实现要点：**再增加两个条件变量**。

<!--
| 条件变量 | 等待者 | 由谁唤醒 |
|---|---|---|
| `not_empty` | 空闲的工作线程 | `tp2_submit` 入队后 |
| `not_full` | 被阻塞的提交者 | 工作线程取走一个任务、腾出一个槽位后 |
| `all_done` | 调用 `tp2_wait` 的线程 | 工作线程执行完任务、且发现队列已空、无任务在执行时 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">条件变量</th>
      <th style="text-align: left;">等待者</th>
      <th style="text-align: left;">由谁唤醒</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>not_empty</code></td>
      <td style="text-align: left;">空闲的工作线程</td>
      <td style="text-align: left;"><code>tp2_submit</code> 入队后</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>not_full</code></td>
      <td style="text-align: left;">被阻塞的提交者</td>
      <td style="text-align: left;">工作线程取走一个任务、腾出一个槽位后</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>all_done</code></td>
      <td style="text-align: left;">调用 <code>tp2_wait</code> 的线程</td>
      <td style="text-align: left;">工作线程执行完任务、且发现队列已空、无任务在执行时</td>
    </tr>
  </tbody>
</table>


> **⭐ 最容易写错的地方**：判断"线程池是否已空闲"时，只检查队列是否为空是**不够**的。任务一旦出队就不在队列里了，但它还在执行。因此必须再维护一个 `running` 计数，"空闲"的判定条件是 `queued == 0 && running == 0`。若漏掉 `running`，`tp2_wait()` 会在最后一批任务尚未算完时提前返回——本节骨架若不补全该逻辑，校验将报 `FAIL`。

### 12.1 接口


In [ ]:
%%writefile src_tp/threadpool2.h
/* ==========================================================================
 * threadpool2.h -- Extension: a bounded queue and a completion barrier.
 *
 * Two limitations of the basic pool are removed here.
 *
 * 1. The basic pool has no way to say "wait until everything I submitted has
 *    finished". The only completion guarantee it offers is tp_destroy(), so a
 *    program that renders one image after another has to build and tear down
 *    the pool every time. tp2_wait() provides that guarantee without
 *    destroying anything.
 *
 * 2. The basic queue is unbounded. A producer that runs faster than the
 *    workers makes it grow without limit, and the memory it occupies is the
 *    only thing that stops it. tp2_submit() blocks while the queue is full,
 *    which is the standard producer-consumer backpressure mechanism.
 * ========================================================================== */
#ifndef THREADPOOL2_H
#define THREADPOOL2_H

typedef struct tp2_pool tp2_pool_t;
typedef void (*tp2_task_fn)(void *arg);

/* capacity <= 0 means "unbounded", i.e. the behaviour of the basic pool. */
tp2_pool_t *tp2_create(int nthreads, int capacity);

/* Append one task. Blocks while the queue holds `capacity` tasks.
 * Returns 0 on success, -1 if the pool is shutting down. */
int tp2_submit(tp2_pool_t *pool, tp2_task_fn fn, void *arg);

/* Return once every task submitted so far has finished running.
 * The pool stays usable afterwards. Intended for a single submitting thread. */
void tp2_wait(tp2_pool_t *pool);

/* Drain, join, release. */
void tp2_destroy(tp2_pool_t *pool);

/* Index of the calling worker, 0 .. nthreads-1; -1 outside the pool. */
int tp2_worker_id(void);

/* Largest number of tasks the queue ever held. */
long tp2_peak_queue(const tp2_pool_t *pool);

#endif /* THREADPOOL2_H */

### 12.2 待补全的实现

请完成文件中的两组 TODO：

- **TODO 1（a、b）**：有界队列的背压——`tp2_submit` 中的等待循环，以及工作线程取走任务后对 `not_full` 的唤醒；
- **TODO 2（a、b、c）**：完成栅栏——`running` 计数的维护、空闲时对 `all_done` 的广播，以及 `tp2_wait` 中的等待循环。

未补全时程序可以正常编译运行，但方案 B、C 会报 `FAIL`。补全后三行应全部 `PASS`。


In [ ]:
%%writefile src_tp/threadpool2_skeleton.c
/* ==========================================================================
 * threadpool2_skeleton.c -- Extension exercise.
 *
 * Complete the two TODO groups below. Everything else is already written.
 *
 *   TODO 1  bounded queue : make tp2_submit() block while the queue is full,
 *                           and make a worker wake one blocked producer after
 *                           it removes a task.
 *   TODO 2  completion    : make tp2_wait() return exactly when every task
 *                           submitted so far has finished running.
 *
 * The program compiles and runs without the TODOs, and reports FAIL: the
 * image is read before all of its pixels have been computed.
 * ========================================================================== */
#include "threadpool2.h"

#include <pthread.h>
#include <stdlib.h>

typedef struct tp2_task {
  tp2_task_fn fn;
  void *arg;
  struct tp2_task *next;
} tp2_task_t;

struct tp2_pool {
  pthread_mutex_t lock;
  pthread_cond_t not_empty;  /* a task is available                          */
  pthread_cond_t not_full;   /* a queue slot became free                     */
  pthread_cond_t all_done;   /* queued == 0 and running == 0                 */

  tp2_task_t *head, *tail;
  long queued;               /* tasks waiting in the queue                   */
  long running;              /* tasks currently being executed by a worker   */
  long peak;                 /* high-water mark of queued                    */
  int capacity;              /* 0 = unbounded                                */
  int shutdown;

  int nthreads;
  pthread_t *threads;
};

static __thread int tp2_tls_id = -1;

int tp2_worker_id(void) { return tp2_tls_id; }

typedef struct {
  tp2_pool_t *pool;
  int id;
} tp2_arg_t;

static void *tp2_worker(void *raw) {
  tp2_arg_t *wa = (tp2_arg_t *)raw;
  tp2_pool_t *p = wa->pool;
  tp2_tls_id = wa->id;
  free(wa);

  for (;;) {
    pthread_mutex_lock(&p->lock);

    while (p->queued == 0 && !p->shutdown)
      pthread_cond_wait(&p->not_empty, &p->lock);

    if (p->queued == 0 && p->shutdown) {
      pthread_mutex_unlock(&p->lock);
      break;
    }

    tp2_task_t *t = p->head;
    p->head = t->next;
    if (p->head == NULL) p->tail = NULL;
    p->queued--;

    /* TODO 1 (b): one queue slot has just become free. Wake at most one
     * producer that may be blocked inside tp2_submit(). One line.
     * Think about why signal is correct here and broadcast is wasteful. */

    /* TODO 2 (a): the task has left the queue but has NOT finished running.
     * Record that fact, so that tp2_wait() cannot return too early. One line. */

    pthread_mutex_unlock(&p->lock);

    t->fn(t->arg);
    free(t);

    pthread_mutex_lock(&p->lock);
    /* TODO 2 (b): the task has finished. Update the counter from TODO 2 (a),
     * and wake everybody waiting in tp2_wait() once the pool has become
     * completely idle. Three lines.
     * Hint: "idle" means p->queued == 0 AND p->running == 0. Testing
     * p->queued alone would let tp2_wait() return while the last tasks were
     * still executing. Use broadcast on p->all_done. */
    pthread_mutex_unlock(&p->lock);
  }
  return NULL;
}

tp2_pool_t *tp2_create(int nthreads, int capacity) {
  if (nthreads <= 0) return NULL;
  tp2_pool_t *p = (tp2_pool_t *)calloc(1, sizeof(tp2_pool_t));
  if (!p) return NULL;

  p->nthreads = nthreads;
  p->capacity = capacity > 0 ? capacity : 0;
  p->threads = (pthread_t *)calloc((size_t)nthreads, sizeof(pthread_t));
  if (!p->threads) {
    free(p);
    return NULL;
  }

  pthread_mutex_init(&p->lock, NULL);
  pthread_cond_init(&p->not_empty, NULL);
  pthread_cond_init(&p->not_full, NULL);
  pthread_cond_init(&p->all_done, NULL);

  for (int i = 0; i < nthreads; i++) {
    tp2_arg_t *wa = (tp2_arg_t *)malloc(sizeof(tp2_arg_t));
    wa->pool = p;
    wa->id = i;
    if (pthread_create(&p->threads[i], NULL, tp2_worker, wa) != 0) {
      free(wa);
      p->nthreads = i;
      tp2_destroy(p);
      return NULL;
    }
  }
  return p;
}

int tp2_submit(tp2_pool_t *pool, tp2_task_fn fn, void *arg) {
  if (!pool || !fn) return -1;

  tp2_task_t *t = (tp2_task_t *)malloc(sizeof(tp2_task_t));
  if (!t) return -1;
  t->fn = fn;
  t->arg = arg;
  t->next = NULL;

  pthread_mutex_lock(&pool->lock);

  /* TODO 1 (a): backpressure. While the queue already holds `capacity` tasks
   * -- and the pool is not shutting down -- wait on pool->not_full.
   * Two lines. capacity == 0 means "unbounded", so do not block in that case.
   * Remember: WHILE, not IF. */

  if (pool->shutdown) {
    pthread_mutex_unlock(&pool->lock);
    free(t);
    return -1;
  }

  if (pool->tail) pool->tail->next = t;
  else pool->head = t;
  pool->tail = t;
  pool->queued++;
  if (pool->queued > pool->peak) pool->peak = pool->queued;

  pthread_cond_signal(&pool->not_empty);
  pthread_mutex_unlock(&pool->lock);
  return 0;
}

void tp2_wait(tp2_pool_t *pool) {
  if (!pool) return;
  pthread_mutex_lock(&pool->lock);
  /* TODO 2 (c): block until the pool is idle, i.e. until nothing is queued
   * and nothing is running. Two lines. */
  pthread_mutex_unlock(&pool->lock);
}

void tp2_destroy(tp2_pool_t *pool) {
  if (!pool) return;

  pthread_mutex_lock(&pool->lock);
  pool->shutdown = 1;
  pthread_cond_broadcast(&pool->not_empty);
  pthread_cond_broadcast(&pool->not_full);   /* release blocked producers */
  pthread_mutex_unlock(&pool->lock);

  for (int i = 0; i < pool->nthreads; i++)
    pthread_join(pool->threads[i], NULL);

  pthread_mutex_destroy(&pool->lock);
  pthread_cond_destroy(&pool->not_empty);
  pthread_cond_destroy(&pool->not_full);
  pthread_cond_destroy(&pool->all_done);
  free(pool->threads);
  free(pool);
}

long tp2_peak_queue(const tp2_pool_t *pool) { return pool ? pool->peak : 0; }

### 12.3 驱动程序

驱动程序把同一幅图像**连续渲染多轮**，这是真实应用的常见形态（一帧接一帧、一批接一批）。三种方案的差异如下：

<!--
| 方案 | 线程池 | 轮次边界 | 队列 |
|---|---|---|---|
| A | 每轮重建 | `tp2_destroy` | 无上限 |
| B | 全程复用 | `tp2_wait` | 无上限 |
| C | 全程复用 | `tp2_wait` | 有上限（默认 $2N$） |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">方案</th>
      <th style="text-align: left;">线程池</th>
      <th style="text-align: left;">轮次边界</th>
      <th style="text-align: left;">队列</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">A</td>
      <td style="text-align: left;">每轮重建</td>
      <td style="text-align: left;"><code>tp2_destroy</code></td>
      <td style="text-align: left;">无上限</td>
    </tr>
    <tr>
      <td style="text-align: left;">B</td>
      <td style="text-align: left;">全程复用</td>
      <td style="text-align: left;"><code>tp2_wait</code></td>
      <td style="text-align: left;">无上限</td>
    </tr>
    <tr>
      <td style="text-align: left;">C</td>
      <td style="text-align: left;">全程复用</td>
      <td style="text-align: left;"><code>tp2_wait</code></td>
      <td style="text-align: left;">有上限（默认 $2N$）</td>
    </tr>
  </tbody>
</table>


**A 与 B 的差异**衡量池复用的收益：单轮耗时越短、线程数越多，收益越显著。**B 与 C 的差异**衡量背压的代价，而 `PeakQueue` 一列显示它换来了什么——队列的峰值长度从"任务总数"降到"容量"。


In [ ]:
%%writefile src_tp/tp_ext.c
/* ==========================================================================
 * tp_ext.c -- Driver for the extension pool (threadpool2).
 *
 * The same Mandelbrot workload is rendered REPEATEDLY, which is what a real
 * application does: a frame, a request, a batch -- one after another.
 *
 *   A. Pool per round        the basic pool's only completion guarantee is
 *                            tp_destroy(), so the whole pool is rebuilt for
 *                            every round
 *   B. One pool + tp2_wait   the pool is built once; tp2_wait() ends a round
 *   C. B + bounded queue     same, but the queue holds at most `cap` tasks,
 *                            so the submitting thread is throttled instead of
 *                            letting the queue grow to the task count
 *
 * A vs B measures what pool reuse is worth. B vs C measures what backpressure
 * costs, and the peak queue length shows what it buys.
 * ========================================================================== */
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <unistd.h>

#include "threadpool2.h"

#define ROUNDS 8      // How many times the image is rendered per method
#define MAX_THREADS 256

#define VIEW_X0 (-2.0)
#define VIEW_Y0 (-2.1)
#define VIEW_SPAN (3.0)

double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static int g_W, g_H, g_maxiter;

static void mandel_range(uint16_t *img, long p0, long p1) {
  const double dx = VIEW_SPAN / (double)g_W;
  const double dy = VIEW_SPAN / (double)g_H;
  for (long p = p0; p < p1; p++) {
    int py = (int)(p / g_W);
    int px = (int)(p % g_W);
    double cr = VIEW_X0 + ((double)px + 0.5) * dx;
    double ci = VIEW_Y0 + ((double)py + 0.5) * dy;
    double zr = 0.0, zi = 0.0;
    int it = 0;
    while (it < g_maxiter) {
      double zr2 = zr * zr, zi2 = zi * zi;
      if (zr2 + zi2 > 4.0) break;
      zi = 2.0 * zr * zi + ci;
      zr = zr2 - zi2 + cr;
      it++;
    }
    img[p] = (uint16_t)it;
  }
}

typedef struct {
  uint16_t *img;
  long p0, p1;
} range_t;

static void mandel_task(void *arg) {
  range_t *r = (range_t *)arg;
  mandel_range(r->img, r->p0, r->p1);
}

static long build_tasks(range_t *tasks, uint16_t *img, long npix, long grain) {
  long n = 0;
  for (long p = 0; p < npix; p += grain) {
    tasks[n].img = img;
    tasks[n].p0 = p;
    tasks[n].p1 = (p + grain < npix) ? p + grain : npix;
    n++;
  }
  return n;
}

static void report(const char *name, double t, double t_base, double npix,
                   long peak, const char *chk) {
  printf("| %-24s | %9.3f | %5.2f x | %7.2f | %8ld | %5s |\n", name, t,
         t_base / t, npix / (t * 1000.0), peak, chk);
}

int main(int argc, char **argv) {
  if (argc < 4) {
    printf("Usage: %s <W> <H> <maxiter> [threads] [grain] [queue_capacity]\n",
           argv[0]);
    return 1;
  }
  g_W = atoi(argv[1]);
  g_H = atoi(argv[2]);
  g_maxiter = atoi(argv[3]);
  int threads = (argc > 4) ? atoi(argv[4]) : 0;
  long grain = (argc > 5) ? atol(argv[5]) : 0;
  int cap = (argc > 6) ? atoi(argv[6]) : 0;

  if (g_W <= 0 || g_H <= 0 || g_maxiter <= 0) return 1;
  if (g_maxiter > 65535) g_maxiter = 65535;
  if (threads <= 0) threads = (int)sysconf(_SC_NPROCESSORS_ONLN);
  if (threads > MAX_THREADS) threads = MAX_THREADS;
  if (grain <= 0) grain = g_W;
  if (cap <= 0) cap = 2 * threads;

  long npix = (long)g_W * (long)g_H;
  long ntasks = (npix + grain - 1) / grain;

  printf("============================================================\n");
  printf(" Extension: completion barrier and bounded queue\n");
  printf(" Image  : %d x %d, max iterations %d\n", g_W, g_H, g_maxiter);
  printf(" Threads: %d   Grain: %ld px  ->  %ld tasks   Rounds: %d\n", threads,
         grain, ntasks, ROUNDS);
  printf(" Bounded queue capacity: %d tasks\n", cap);
  printf("============================================================\n");

  size_t bytes = (size_t)npix * sizeof(uint16_t);
  uint16_t *img_ref = (uint16_t *)malloc(bytes);
  uint16_t *img = (uint16_t *)malloc(bytes);
  range_t *tasks = (range_t *)malloc(sizeof(range_t) * (size_t)(ntasks + 1));
  if (!img_ref || !img || !tasks) return 1;

  memset(img_ref, 0, bytes);
  mandel_range(img_ref, 0, npix);
  long nt = build_tasks(tasks, img, npix, grain);

  double start, end;
  long peak;
  const char *chk;

  printf("\n---------------------------------------------------------------------------\n");
  printf("| %-24s | %9s | %7s | %7s | %8s | %5s |\n", "Method", "Time (ms)",
         "Speedup", "Mpix/s", "PeakQueue", "Check");
  printf("|--------------------------|-----------|---------|---------|----------|-------|\n");

  /* A. one pool per round -- the basic pool's pattern */
  peak = 0;
  start = get_time_ms();
  for (int r = 0; r < ROUNDS; r++) {
    memset(img, 0, bytes);   // start every round from a cleared image
    tp2_pool_t *pool = tp2_create(threads, 0);
    if (!pool) return 1;
    for (long i = 0; i < nt; i++) tp2_submit(pool, mandel_task, &tasks[i]);
    long pk = tp2_peak_queue(pool);   /* read BEFORE the object is destroyed */
    if (pk > peak) peak = pk;
    tp2_destroy(pool);
  }
  end = get_time_ms();
  double t_a = (end - start) / ROUNDS;
  report("A. Pool per round", t_a, t_a, (double)npix, peak,
         memcmp(img_ref, img, bytes) == 0 ? "PASS" : "FAIL");

  /* B. one pool for all rounds, tp2_wait ends each round */
  {
    tp2_pool_t *pool = tp2_create(threads, 0);
    if (!pool) return 1;
    start = get_time_ms();
    for (int r = 0; r < ROUNDS; r++) {
      memset(img, 0, bytes);
      for (long i = 0; i < nt; i++) tp2_submit(pool, mandel_task, &tasks[i]);
      tp2_wait(pool);   /* the round is over exactly here */
    }
    end = get_time_ms();
    peak = tp2_peak_queue(pool);
    /* Check the image while the pool is still alive. Checking after
     * tp2_destroy() would prove nothing: destroying the pool drains the
     * queue, so the image would be complete even if tp2_wait() did not
     * actually wait. */
    chk = memcmp(img_ref, img, bytes) == 0 ? "PASS" : "FAIL";
    tp2_destroy(pool);
  }
  double t_b = (end - start) / ROUNDS;
  report("B. One pool + tp2_wait", t_b, t_a, (double)npix, peak, chk);

  /* C. same, with a bounded queue */
  {
    tp2_pool_t *pool = tp2_create(threads, cap);
    if (!pool) return 1;
    start = get_time_ms();
    for (int r = 0; r < ROUNDS; r++) {
      memset(img, 0, bytes);
      for (long i = 0; i < nt; i++) tp2_submit(pool, mandel_task, &tasks[i]);
      tp2_wait(pool);
    }
    end = get_time_ms();
    peak = tp2_peak_queue(pool);
    chk = memcmp(img_ref, img, bytes) == 0 ? "PASS" : "FAIL";
    tp2_destroy(pool);
  }
  double t_c = (end - start) / ROUNDS;
  report("C. B + bounded queue", t_c, t_a, (double)npix, peak, chk);

  printf("---------------------------------------------------------------------------\n");
  printf("  PeakQueue is the largest number of tasks the queue ever held. In A and\n");
  printf("  B the queue is unbounded, so it grows until the producer runs out of\n");
  printf("  tasks; in C it never exceeds the capacity, at the price of blocking\n");
  printf("  the submitting thread.\n");

  free(img_ref);
  free(img);
  free(tasks);
  return 0;
}

In [ ]:
# 编译运行：TODO 未补全时，B、C 两行应当报 FAIL
BE = compile_c("src_tp/tp_ext.c src_tp/threadpool2_skeleton.c", "src_tp/tp_ext")
# 参数：宽 高 最大迭代次数 线程数 粒度 队列容量
out_ext = run_bin(BE, 512, 512, 500, NT, 512, 2 * NT)

## 13. 🔧 动手练习

1. **补全扩展实验的两组 TODO**，直到 12.3 的三行全部 `PASS`。随后回答：把 `tp2_submit` 中的 `while` 改成 `if` 会怎样？为什么"另一个生产者抢走了刚腾出的槽位"这一情形在单生产者程序中不会出现，而该写法仍然必须是 `while`？

2. **测量线程创建的单次成本**。写一个程序，循环 $10^4$ 次 `pthread_create` + `pthread_join`（线程体为空），得到单次创建回收的平均耗时 $c$。用它估算第 6 节中 `Spawn per Task` 一行的理论开销 $\text{任务数} \times c / N$，与实测的两行差值比较，说明差额来自何处。

3. **改变任务的划分方向**。当前任务是连续的像素区间；改为**交错划分**（第 $i$ 个任务负责 $p \equiv i \pmod{K}$ 的像素），在**不使用动态队列**的前提下重测 `Pool + Static` 的不均衡因子。这种做法能否替代动态队列？代价是什么？（提示：考虑访存的空间局部性。）

4. **给线程池加上任务计数**。在 `threadpool.c` 中增加"已完成任务数"统计。请分别用（a）一个受互斥锁保护的全局计数器、（b）每线程一个计数器最后求和 两种方式实现，在粒度为 1 的极端情形下比较两者的耗时，并结合第四章伪共享实验解释差异。

5. **观察惊群效应**。把 `tp_submit` 中的 `pthread_cond_signal` 改为 `pthread_cond_broadcast`，在细粒度（如 grain = 16）、大线程数下重测。差异有多大？为什么在粗粒度下几乎测不出差异？


## 14. 🤔 思考题

1. `tp_worker` 在执行任务之前释放了互斥锁。假如把 `pthread_mutex_unlock` 移到 `t->fn(t->arg)` 之后，程序仍然能得到正确的图像吗？耗时会变成多少？请先预测，再动手验证。

2. `tp_destroy` 使用 `broadcast` 唤醒全部工作线程。若改成在循环中调用 $N$ 次 `signal`，是否等价？（提示：考虑此时有工作线程正在执行任务、尚未回到等待状态的情形。）

3. 本实验的任务队列是 FIFO。若改成 LIFO（后进先出），正确性会受影响吗？在什么样的任务结构下 LIFO 反而更优？（提示：考虑递归产生子任务的场景与缓存局部性。）

4. `tp_submit` 在队列无上限时永不阻塞。设想一个流式场景：生产者每秒提交 $10^5$ 个任务，而线程池每秒只能完成 $10^4$ 个。程序会以何种方式失败？失败前有哪些可观测的征兆？

5. 假设某任务体内部调用了 `tp_submit` 向同一个线程池提交新任务。这样做是否安全？若队列有上限，会发生什么？（提示：这是有界队列线程池的经典死锁场景。）

6. 第 9 节的最优粒度是在**当前线程数**下测得的。若线程数翻倍，最优粒度应当变大还是变小？请从 $T(g)$ 的两项分别推导，再用实验验证。


## 15. 小结与后续

本实验把第四章的单项技术组装成了一个可复用的构件，并用实验回答了四个问题：

<!--
| 问题 | 答案 | 证据 |
|---|---|---|
| 为什么需要线程池 | 线程创建与回收是内核开销，且分批 `join` 引入了人为同步点 | 第 6 节方案 2 与方案 4 的对比 |
| 为什么需要任务队列 | 队列使"先完成的线程先领取新任务"成为可能，这本身就是动态负载均衡 | 第 8 节两个方案的不均衡因子 |
| 任务粒度如何选取 | 过细则队列与锁开销主导，过粗则负载不均衡主导，$T(g)$ 为下凹曲线 | 第 9 节的粒度扫描 |
| 哪些写法是错的 | 条件等待必须用 `while`；关闭判定的书写顺序决定 drain 还是 stop-now | 第 10 节的两次故障注入 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">问题</th>
      <th style="text-align: left;">答案</th>
      <th style="text-align: left;">证据</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">为什么需要线程池</td>
      <td style="text-align: left;">线程创建与回收是内核开销，且分批 <code>join</code> 引入了人为同步点</td>
      <td style="text-align: left;">第 6 节方案 2 与方案 4 的对比</td>
    </tr>
    <tr>
      <td style="text-align: left;">为什么需要任务队列</td>
      <td style="text-align: left;">队列使"先完成的线程先领取新任务"成为可能，这本身就是动态负载均衡</td>
      <td style="text-align: left;">第 8 节两个方案的不均衡因子</td>
    </tr>
    <tr>
      <td style="text-align: left;">任务粒度如何选取</td>
      <td style="text-align: left;">过细则队列与锁开销主导，过粗则负载不均衡主导，$T(g)$ 为下凹曲线</td>
      <td style="text-align: left;">第 9 节的粒度扫描</td>
    </tr>
    <tr>
      <td style="text-align: left;">哪些写法是错的</td>
      <td style="text-align: left;">条件等待必须用 <code>while</code>；关闭判定的书写顺序决定 drain 还是 stop-now</td>
      <td style="text-align: left;">第 10 节的两次故障注入</td>
    </tr>
  </tbody>
</table>


### 三条一般性结论

1. **加速比的上限由不均衡因子决定**：$S \le N / \text{imbalance}$。在归因于内存带宽之前，先把这个数测出来。
2. **静默失败比崩溃更危险**。注入二的程序退出码为 0、无任何警告，只有结果是错的。因此每一个并行版本都必须附带可执行的正确性校验。
3. **同步原语的选择要有理由**。`signal` 还是 `broadcast`、`while` 还是 `if`、先判队列还是先判标志——每一处都对应一个明确的语义，而不是风格偏好。

### 与其他章节的联系

<!--
| 章节 | 关系 |
|---|---|
| 第四章 实验六 · 生产者—消费者 | 线程池就是一个多消费者的生产者—消费者队列；扩展实验的有界队列与该实验完全同构 |
| 第四章 实验九 · 伪共享 | 本实验的每线程统计计数器按缓存行填充，正是该实验结论的直接应用 |
| 第五章 · OpenMP | OpenMP 运行时内部同样维护线程池，`schedule(static)` 与 `schedule(dynamic)` 对应本实验的方案 3 与方案 4。学完本实验后，`schedule` 子句的选择就不再是记忆题 |
| 第三章 · NEON | 本实验的任务体是标量代码。若把 Mandelbrot 内层迭代改写为 NEON 同时处理 4 个像素，即可在 TLP 之上再叠加 DLP |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">章节</th>
      <th style="text-align: left;">关系</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">第四章 实验六 · 生产者—消费者</td>
      <td style="text-align: left;">线程池就是一个多消费者的生产者—消费者队列；扩展实验的有界队列与该实验完全同构</td>
    </tr>
    <tr>
      <td style="text-align: left;">第四章 实验九 · 伪共享</td>
      <td style="text-align: left;">本实验的每线程统计计数器按缓存行填充，正是该实验结论的直接应用</td>
    </tr>
    <tr>
      <td style="text-align: left;">第五章 · OpenMP</td>
      <td style="text-align: left;">OpenMP 运行时内部同样维护线程池，<code>schedule(static)</code> 与 <code>schedule(dynamic)</code> 对应本实验的方案 3 与方案 4。学完本实验后，<code>schedule</code> 子句的选择就不再是记忆题</td>
    </tr>
    <tr>
      <td style="text-align: left;">第三章 · NEON</td>
      <td style="text-align: left;">本实验的任务体是标量代码。若把 Mandelbrot 内层迭代改写为 NEON 同时处理 4 个像素，即可在 TLP 之上再叠加 DLP</td>
    </tr>
  </tbody>
</table>

